In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 1
# Drive + Research Paths
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive")
RESEARCH = BASE / "maxmin_research"

print("=" * 60)
print("EXPERIMENT 003 — MLDR HINDI")
print("=" * 60)

print("BASE:", BASE)
print("RESEARCH:", RESEARCH)
print("Research folder exists:", RESEARCH.exists())

CKPT_DIR = RESEARCH / "cache" / "mldr003_maxmin_checkpoints"
print("Checkpoint directory:", CKPT_DIR)
print("Checkpoint directory exists:", CKPT_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXPERIMENT 003 — MLDR HINDI
BASE: /content/drive/MyDrive
RESEARCH: /content/drive/MyDrive/maxmin_research
Research folder exists: True
Checkpoint directory: /content/drive/MyDrive/maxmin_research/cache/mldr003_maxmin_checkpoints
Checkpoint directory exists: True


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 2
# Verify Existing Max–Min Checkpoints
# ============================================================

import json
import pickle

MANIFEST_PATH = CKPT_DIR / "manifest.json"

print("=" * 60)
print("CHECKPOINT INTEGRITY CHECK")
print("=" * 60)

print("Manifest exists:", MANIFEST_PATH.exists())

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed = set(manifest["completed_docids"])

checkpoint_files = sorted(CKPT_DIR.glob("batch_*.pkl"))

print("Completed documents:", len(completed))
print("Checkpoint files:", len(checkpoint_files))

print("\nFirst checkpoints:")
for p in checkpoint_files[:3]:
    print(" ", p.name)

print("\nLast checkpoints:")
for p in checkpoint_files[-3:]:
    print(" ", p.name)

# Verify the final checkpoint can actually be opened
last_checkpoint = checkpoint_files[-1]

with open(last_checkpoint, "rb") as f:
    last_data = pickle.load(f)

print("\nLast checkpoint:", last_checkpoint.name)
print("Documents inside last checkpoint:", len(last_data))

print("\nExpected:")
print("  Completed documents: 1900")
print("  Last checkpoint: batch_1900.pkl")

CHECKPOINT INTEGRITY CHECK
Manifest exists: True
Completed documents: 1900
Checkpoint files: 19

First checkpoints:
  batch_0100.pkl
  batch_0200.pkl
  batch_0300.pkl

Last checkpoints:
  batch_1700.pkl
  batch_1800.pkl
  batch_1900.pkl

Last checkpoint: batch_1900.pkl
Documents inside last checkpoint: 100

Expected:
  Completed documents: 1900
  Last checkpoint: batch_1900.pkl


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 3
# Imports + Reproducibility
# ============================================================

import os
import gc
import re
import json
import pickle
import time
import math
import random

import numpy as np
import pandas as pd
import torch

from pathlib import Path

SEED = 20261003

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("ENVIRONMENT READY")
print("=" * 60)

print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ENVIRONMENT READY
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 4
# MLDR Hindi — Hugging Face Setup
# ============================================================

!pip -q install -U huggingface_hub

from huggingface_hub import hf_hub_download

HF_REPO = "Shitao/MLDR"
HF_REVISION = "main"

MLDR_FILES = {
    "corpus": "mldr-v1.0-hi/corpus.jsonl.gz",
    "dev": "mldr-v1.0-hi/dev.jsonl.gz",
    "test": "mldr-v1.0-hi/test.jsonl.gz",
    "dev_qrels": "qrels/qrels.mldr-v1.0-hi-dev.tsv",
    "test_qrels": "qrels/qrels.mldr-v1.0-hi-test.tsv",
}

print("=" * 60)
print("MLDR HINDI — HUGGING FACE SETUP")
print("=" * 60)

print("Repository:", HF_REPO)
print("Files:")
for name, path in MLDR_FILES.items():
    print(f"  {name}: {path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
MLDR HINDI — HUGGING FACE SETUP
Repository: Shitao/MLDR
Files:
  corpus: mldr-v1.0-hi/corpus.jsonl.gz
  dev: mldr-v1.0-hi/dev.jsonl.gz
  test: mldr-v1.0-hi/test.jsonl.gz
  dev_qrels: qrels/qrels.mldr-v1.0-hi-dev.tsv
  test_qrels: qrels/qrels.mldr-v1.0-hi-test.tsv


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 5
# Download MLDR Hindi Files
# ============================================================

print("=" * 60)
print("DOWNLOADING MLDR HINDI")
print("=" * 60)

MLDR_LOCAL = {}

for name, hf_path in MLDR_FILES.items():
    print(f"\nDownloading: {name}")
    print(f"  {hf_path}")

    local_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=hf_path,
        revision=HF_REVISION
    )

    MLDR_LOCAL[name] = Path(local_path)

    print(f"  Saved locally: {local_path}")

print("\n" + "=" * 60)
print("DOWNLOAD COMPLETE")
print("=" * 60)

for name, path in MLDR_LOCAL.items():
    print(f"{name:12s}: {path}")

DOWNLOADING MLDR HINDI

Downloading: corpus
  mldr-v1.0-hi/corpus.jsonl.gz


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6ac6717f-1a3483dd69c5fa5063bb1a1c;9e14a071-b680-4ec5-99b2-3c7ef95309cd)

Repository Not Found for url: https://huggingface.co/Shitao/MLDR/resolve/main/mldr-v1.0-hi/corpus.jsonl.gz.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 5A
# Restore Colab-compatible Hugging Face Hub
# ============================================================

!pip -q install "huggingface_hub<2.0,>=0.25.0"

print("Hugging Face Hub compatibility version installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 2.6 MB/s eta 0:00:00
Hugging Face Hub compatibility version installed.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 5B
# Download MLDR Hindi Files
# ============================================================

from huggingface_hub import hf_hub_download

MLDR_LOCAL = {}

for name, hf_path in MLDR_FILES.items():
    print(f"\nDownloading: {name}")

    local_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=hf_path,
        revision=HF_REVISION
    )

    MLDR_LOCAL[name] = Path(local_path)

    print(f"Saved: {local_path}")

print("\n" + "=" * 60)
print("DOWNLOAD COMPLETE")
print("=" * 60)

for name, path in MLDR_LOCAL.items():
    print(f"{name:12s}: {path}")


Downloading: corpus


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6ac671ab-718dc3fd1c691c3566c8a456;80f281f9-d777-4303-b954-140e747b44fc)

Repository Not Found for url: https://huggingface.co/Shitao/MLDR/resolve/main/mldr-v1.0-hi/corpus.jsonl.gz.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 5C
# Direct MLDR Hindi Download
# ============================================================

import os
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
MLDR_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://huggingface.co/datasets/Shitao/MLDR/resolve/main"

FILES = {
    "corpus": "mldr-v1.0-hi/corpus.jsonl.gz",
    "dev": "mldr-v1.0-hi/dev.jsonl.gz",
    "test": "mldr-v1.0-hi/test.jsonl.gz",
    "dev_qrels": "qrels/qrels.mldr-v1.0-hi-dev.tsv",
    "test_qrels": "qrels/qrels.mldr-v1.0-hi-test.tsv",
}

MLDR_LOCAL = {}

for name, hf_path in FILES.items():
    filename = Path(hf_path).name
    output = MLDR_DIR / filename

    print(f"\nDownloading {name}...")
    print(hf_path)

    url = f"{BASE_URL}/{hf_path}?download=true"

    result = os.system(
        f'curl -L --fail --retry 3 "{url}" -o "{output}"'
    )

    if result != 0:
        raise RuntimeError(f"Download failed: {name}")

    MLDR_LOCAL[name] = output
    print("Saved:", output)
    print("Size:", output.stat().st_size / (1024**2), "MB")

print("\n" + "=" * 60)
print("MLDR DOWNLOAD COMPLETE")
print("=" * 60)

for name, path in MLDR_LOCAL.items():
    print(f"{name:12s}: {path}")


mldr-v1.0-hi/corpus.jsonl.gz
Saved: /content/mldr003/corpus.jsonl.gz
Size: 35.1596622467041 MB

mldr-v1.0-hi/dev.jsonl.gz
Saved: /content/mldr003/dev.jsonl.gz
Size: 9.688947677612305 MB

mldr-v1.0-hi/test.jsonl.gz
Saved: /content/mldr003/test.jsonl.gz
Size: 9.839764595031738 MB

qrels/qrels.mldr-v1.0-hi-dev.tsv
Saved: /content/mldr003/qrels.mldr-v1.0-hi-dev.tsv
Size: 0.004921913146972656 MB

qrels/qrels.mldr-v1.0-hi-test.tsv
Saved: /content/mldr003/qrels.mldr-v1.0-hi-test.tsv
Size: 0.004925727844238281 MB

MLDR DOWNLOAD COMPLETE
corpus      : /content/mldr003/corpus.jsonl.gz
dev         : /content/mldr003/dev.jsonl.gz
test        : /content/mldr003/test.jsonl.gz
dev_qrels   : /content/mldr003/qrels.mldr-v1.0-hi-dev.tsv
test_qrels  : /content/mldr003/qrels.mldr-v1.0-hi-test.tsv


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 6
# Load MLDR Hindi Queries + Qrels
# ============================================================

import gzip
import json
import pandas as pd

def load_jsonl_gz(path):
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

dev_records = load_jsonl_gz(MLDR_LOCAL["dev"])
test_records = load_jsonl_gz(MLDR_LOCAL["test"])

dev_qrels = pd.read_csv(
    MLDR_LOCAL["dev_qrels"],
    sep="\t"
)

test_qrels = pd.read_csv(
    MLDR_LOCAL["test_qrels"],
    sep="\t"
)

print("=" * 60)
print("MLDR HINDI — QUERY/QREL AUDIT")
print("=" * 60)

print("DEV records:", len(dev_records))
print("DEV unique query IDs:", len({r["query_id"] for r in dev_records}))
print("TEST records:", len(test_records))
print("TEST unique query IDs:", len({r["query_id"] for r in test_records}))

print("\nDEV qrels:")
print("  Rows:", len(dev_qrels))
print("  Unique qids:", dev_qrels["query_id"].nunique())
print("  Unique docids:", dev_qrels["corpus-id"].nunique())
print("  Labels:", sorted(dev_qrels["score"].unique()))

print("\nTEST qrels:")
print("  Rows:", len(test_qrels))
print("  Unique qids:", test_qrels["query_id"].nunique())
print("  Unique docids:", test_qrels["corpus-id"].nunique())
print("  Labels:", sorted(test_qrels["score"].unique()))

print("\nFirst DEV record:")
print(dev_records[0])

print("\nFirst TEST record:")
print(test_records[0])

MLDR HINDI — QUERY/QREL AUDIT
DEV records: 200
DEV unique query IDs: 200
TEST records: 200
TEST unique query IDs: 200

DEV qrels:
  Rows: 199


KeyError: 'query_id'

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 6A
# Inspect MLDR Qrels Format
# ============================================================

print("=" * 60)
print("QRELS FORMAT")
print("=" * 60)

print("DEV qrels columns:")
print(dev_qrels.columns.tolist())

print("\nDEV qrels shape:", dev_qrels.shape)
print("\nDEV qrels first 10 rows:")
print(dev_qrels.head(10).to_string(index=False))

print("\nTEST qrels columns:")
print(test_qrels.columns.tolist())

print("\nTEST qrels shape:", test_qrels.shape)
print("\nTEST qrels first 10 rows:")
print(test_qrels.head(10).to_string(index=False))

QRELS FORMAT
DEV qrels columns:
['q-hi-9', 'Q0', 'doc-hi-72', '1']

DEV qrels shape: (199, 4)

DEV qrels first 10 rows:
 q-hi-9 Q0  doc-hi-72  1
q-hi-19 Q0 doc-hi-151  1
q-hi-21 Q0 doc-hi-167  1
q-hi-27 Q0 doc-hi-214  1
q-hi-38 Q0 doc-hi-301  1
q-hi-41 Q0 doc-hi-324  1
q-hi-42 Q0 doc-hi-332  1
q-hi-47 Q0 doc-hi-372  1
q-hi-56 Q0 doc-hi-444  1
q-hi-57 Q0 doc-hi-452  1
q-hi-64 Q0 doc-hi-505  1

TEST qrels columns:
['q-hi-1', 'Q0', 'doc-hi-8', '1']

TEST qrels shape: (199, 4)

TEST qrels first 10 rows:
 q-hi-1 Q0   doc-hi-8  1
 q-hi-7 Q0  doc-hi-56  1
q-hi-22 Q0 doc-hi-174  1
q-hi-23 Q0 doc-hi-182  1
q-hi-25 Q0 doc-hi-198  1
q-hi-33 Q0 doc-hi-261  1
q-hi-34 Q0 doc-hi-269  1
q-hi-35 Q0 doc-hi-277  1
q-hi-44 Q0 doc-hi-348  1
q-hi-53 Q0 doc-hi-420  1
q-hi-55 Q0 doc-hi-436  1


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 6B
# Correct MLDR Qrels Loading
# ============================================================

QREL_COLUMNS = ["query_id", "iteration", "doc_id", "relevance"]

dev_qrels = pd.read_csv(
    MLDR_LOCAL["dev_qrels"],
    sep="\t",
    header=None,
    names=QREL_COLUMNS
)

test_qrels = pd.read_csv(
    MLDR_LOCAL["test_qrels"],
    sep="\t",
    header=None,
    names=QREL_COLUMNS
)

print("=" * 60)
print("MLDR HINDI — CORRECTED QREL AUDIT")
print("=" * 60)

for name, qrels in [("DEV", dev_qrels), ("TEST", test_qrels)]:
    print(f"\n{name}:")
    print("  Rows:", len(qrels))
    print("  Unique qids:", qrels["query_id"].nunique())
    print("  Unique docids:", qrels["doc_id"].nunique())
    print("  Relevance labels:", sorted(qrels["relevance"].unique()))

print("\nFirst DEV qrels:")
print(dev_qrels.head(5).to_string(index=False))

print("\nFirst TEST qrels:")
print(test_qrels.head(5).to_string(index=False))

MLDR HINDI — CORRECTED QREL AUDIT

DEV:
  Rows: 200
  Unique qids: 200
  Unique docids: 200
  Relevance labels: [np.int64(1)]

TEST:
  Rows: 200
  Unique qids: 200
  Unique docids: 200
  Relevance labels: [np.int64(1)]

First DEV qrels:
query_id iteration     doc_id  relevance
  q-hi-9        Q0  doc-hi-72          1
 q-hi-19        Q0 doc-hi-151          1
 q-hi-21        Q0 doc-hi-167          1
 q-hi-27        Q0 doc-hi-214          1
 q-hi-38        Q0 doc-hi-301          1

First TEST qrels:
query_id iteration     doc_id  relevance
  q-hi-1        Q0   doc-hi-8          1
  q-hi-7        Q0  doc-hi-56          1
 q-hi-22        Q0 doc-hi-174          1
 q-hi-23        Q0 doc-hi-182          1
 q-hi-25        Q0 doc-hi-198          1


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 7
# Load MLDR Hindi Corpus
# ============================================================

print("=" * 60)
print("LOADING MLDR HINDI CORPUS")
print("=" * 60)

corpus = {}

with gzip.open(MLDR_LOCAL["corpus"], "rt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            corpus[record["docid"]] = record["text"]

print("Documents:", len(corpus))
print("Unique docids:", len(corpus))

print("\nFirst 3 document IDs:")
for docid in list(corpus.keys())[:3]:
    print(" ", docid)

first_docid = next(iter(corpus))
print("\nFirst document:")
print("DocID:", first_docid)
print("Characters:", len(corpus[first_docid]))
print("Preview:", corpus[first_docid][:500])

LOADING MLDR HINDI CORPUS
Documents: 3806
Unique docids: 3806

First 3 document IDs:
  doc-hi-428
  doc-hi-4651
  doc-hi-3203

First document:
DocID: doc-hi-428
Characters: 16649
Preview: उज्जैन
[[चित्र:Mahakal temple.jpg|250px|right|thumb|उज्जैन का प्रसिद्ध महाकालेश्वर मंदिर]]
उज्जैन (उज्जयिनी) भारत के मध्य प्रदेश राज्य का एक प्रमुख शहर है जो क्षिप्रा नदी या शिप्रा नदी के किनारे पर बसा है। यह एक अत्यन्त प्राचीन शहर है। यह महान सम्राट विक्रमादित्य के राज्य की राजधानी थी ।  उज्जैन को कालिदास की नगरी के नाम से भी जाना जाता है। यहाँ हर 12 वर्ष पर सिंहस्थ महाकुंभ मेला लगता है। भगवान शिव के 12 ज्योतिर्लिंगों में एक महाकाल इस नगरी में स्थित है। उज्जैन मध्य प्रदेश के सबसे बड़े शहर इन्दौ


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 8
# Corpus / Qrel Integrity Audit
# ============================================================

dev_positive_docs = set(dev_qrels["doc_id"])
test_positive_docs = set(test_qrels["doc_id"])

print("=" * 60)
print("MLDR HINDI — CORPUS INTEGRITY AUDIT")
print("=" * 60)

print("Corpus documents:", len(corpus))

print("\nDEV:")
print("  Queries:", len(dev_records))
print("  Positive docs:", len(dev_positive_docs))
print("  Missing positives:",
      len(dev_positive_docs - set(corpus)))

print("\nTEST:")
print("  Queries:", len(test_records))
print("  Positive docs:", len(test_positive_docs))
print("  Missing positives:",
      len(test_positive_docs - set(corpus)))

print("\nDEV/TEST positive overlap:",
      len(dev_positive_docs & test_positive_docs))

# Query IDs in qrels vs query files
dev_query_ids = {r["query_id"] for r in dev_records}
test_query_ids = {r["query_id"] for r in test_records}

print("\nDEV qrel/query mismatch:")
print("  In qrels but not queries:", len(set(dev_qrels["query_id"]) - dev_query_ids))
print("  In queries but not qrels:", len(dev_query_ids - set(dev_qrels["query_id"])))

print("\nTEST qrel/query mismatch:")
print("  In qrels but not queries:", len(set(test_qrels["query_id"]) - test_query_ids))
print("  In queries but not qrels:", len(test_query_ids - set(test_qrels["query_id"])))

MLDR HINDI — CORPUS INTEGRITY AUDIT
Corpus documents: 3806

DEV:
  Queries: 200
  Positive docs: 200
  Missing positives: 0

TEST:
  Queries: 200
  Positive docs: 200
  Missing positives: 0

DEV/TEST positive overlap: 0

DEV qrel/query mismatch:
  In qrels but not queries: 0
  In queries but not qrels: 0

TEST qrel/query mismatch:
  In qrels but not queries: 0
  In queries but not qrels: 0


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 9
# BGE-M3 Tokenizer Audit
# ============================================================

from transformers import AutoTokenizer

print("=" * 60)
print("LOADING BGE-M3 TOKENIZER")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

docids = list(corpus.keys())

token_lengths = []

for i, docid in enumerate(docids):
    tokens = tokenizer(
        corpus[docid],
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    token_lengths.append(len(tokens))

token_lengths = np.array(token_lengths)

print("\nDocuments:", len(token_lengths))

print("\nToken length statistics:")
for name, value in [
    ("Mean", np.mean(token_lengths)),
    ("Median", np.median(token_lengths)),
    ("P90", np.percentile(token_lengths, 90)),
    ("P95", np.percentile(token_lengths, 95)),
    ("P99", np.percentile(token_lengths, 99)),
    ("Max", np.max(token_lengths)),
]:
    print(f"  {name:7s}: {value:.2f}")

print("\nAbove BGE-M3 limits:")
for limit in [4096, 8192, 12288, 16384, 32768]:
    count = np.sum(token_lengths > limit)
    print(f"  > {limit:5d}: {count:4d} ({100 * count / len(token_lengths):.2f}%)")

docids_token = dict(zip(docids, token_lengths))

print("\nTokenizer audit complete.")

LOADING BGE-M3 TOKENIZER


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8943 > 8192). Running this sequence through the model will result in indexing errors



Documents: 3806

Token length statistics:
  Mean   : 5555.04
  Median : 3565.50
  P90    : 12120.50
  P95    : 15327.00
  P99    : 22126.55
  Max    : 92405.00

Above BGE-M3 limits:
  >  4096: 1801 (47.32%)
  >  8192:  955 (25.09%)
  > 12288:  368 (9.67%)
  > 16384:  148 (3.89%)
  > 32768:    6 (0.16%)

Tokenizer audit complete.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 10
# Relevant Document Token-Length Audit
# ============================================================

dev_positive_docs = set(dev_qrels["doc_id"])
test_positive_docs = set(test_qrels["doc_id"])

dev_lengths = np.array([
    docids_token[docid]
    for docid in dev_positive_docs
])

test_lengths = np.array([
    docids_token[docid]
    for docid in test_positive_docs
])

print("=" * 60)
print("RELEVANT DOCUMENT TOKEN AUDIT")
print("=" * 60)

for name, lengths in [("DEV", dev_lengths), ("TEST", test_lengths)]:
    print(f"\n{name} relevant documents: {len(lengths)}")

    print(f"  Mean   : {np.mean(lengths):.2f}")
    print(f"  Median : {np.median(lengths):.2f}")
    print(f"  P90    : {np.percentile(lengths, 90):.2f}")
    print(f"  P95    : {np.percentile(lengths, 95):.2f}")
    print(f"  P99    : {np.percentile(lengths, 99):.2f}")
    print(f"  Max    : {np.max(lengths):.2f}")

    print("  >4096 :", np.sum(lengths > 4096))
    print("  >8192 :", np.sum(lengths > 8192))

print("\nDEV/TEST relevant-document overlap:",
      len(dev_positive_docs & test_positive_docs))

RELEVANT DOCUMENT TOKEN AUDIT

DEV relevant documents: 200
  Mean   : 6287.63
  Median : 6235.00
  P90    : 7533.40
  P95    : 7700.75
  P99    : 7861.39
  Max    : 7988.00
  >4096 : 200
  >8192 : 0

TEST relevant documents: 200
  Mean   : 6303.10
  Median : 6279.00
  P90    : 7593.90
  P95    : 7782.20
  P99    : 7956.05
  Max    : 7967.00
  >4096 : 200
  >8192 : 0

DEV/TEST relevant-document overlap: 0


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 11
# Sentence Splitter
# ============================================================

SENTENCE_PATTERN = re.compile(r'(?<=[.!?।॥！？])\s+')

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()

    if not text:
        return []

    sentences = SENTENCE_PATTERN.split(text)

    return [s.strip() for s in sentences if s.strip()]

print("=" * 60)
print("SENTENCE SPLITTER")
print("=" * 60)

# Smoke test on documents spanning different lengths
sample_docids = [
    docids[np.argmin(token_lengths)],
    docids[np.argsort(token_lengths)[len(token_lengths)//2]],
    docids[np.argmax(token_lengths)]
]

for docid in sample_docids:
    sentences = split_sentences(corpus[docid])

    print("\nDoc:", docid)
    print("Tokens:", docids_token[docid])
    print("Sentences:", len(sentences))
    print("First sentence:", sentences[0][:200] if sentences else "[EMPTY]")

print("\nSplitter ready.")

SENTENCE SPLITTER

Doc: doc-hi-3375
Tokens: 1001
Sentences: 27
First sentence: रूंबा (नृत्य) thumb|220px|right|रूंबा कोलंबिया, एक अफ्रीकी-क्यूबाई रूंबा रूंबा दो बिल्कुल अलग-अलग अर्थों के साथ नृत्य के लिए प्रयुक्त होने वाला शब्द है।

Doc: doc-hi-4534
Tokens: 3571
Sentences: 125
First sentence: थेरेसा गोह Category:Pages using infobox swimmer with nationality parameter थेरेसा गोह रुई सी बीबीएम ( ; जन्म १६ फरवरी १९८७) सिंगापुर के तैराक और पैरालंपिक पदक विजेता हैं, जिन्होंने २०१६ ग्रीष्मकालीन प

Doc: doc-hi-5111
Tokens: 92405
Sentences: 646
First sentence: उर्दू शायरों की सूची 13 वीं शताब्दी 15वीं शताब्दी 16वीं शताब्दी 17वीं शताब्दी 17वीं शताब्दी भारत भूषण पंत 17वीं शताब्दी 17वीं शताब्दी 17वीं शताब्दी • भारत भूषण पंत 17वीं शताब्दी 17वीं शताब्दी 17वीं शत

Splitter ready.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 12
# Max–Min Segmentation
# ============================================================

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def maxmin_segment(sentences, embeddings,
                   hard_thr=0.6,
                   c=0.9,
                   init_const=1.5):

    n = len(sentences)

    if n == 0:
        return []

    if n == 1:
        return [(0,)]

    chunks = []
    current = [0]

    for i in range(1, n):

        candidate = embeddings[i]

        # ----------------------------------------------------
        # Single-sentence current chunk
        # ----------------------------------------------------
        if len(current) == 1:

            prev = embeddings[current[0]]
            sim = float(np.dot(prev, candidate))

            threshold = hard_thr

            if init_const * sim > threshold:
                current.append(i)
            else:
                chunks.append(tuple(current))
                current = [i]

            continue

        # ----------------------------------------------------
        # Current chunk has >= 2 sentences
        # ----------------------------------------------------
        current_embeddings = embeddings[current]

        # Similarity of candidate to every sentence in chunk
        candidate_sims = current_embeddings @ candidate

        max_sim = float(np.max(candidate_sims))

        # Minimum pairwise similarity inside current chunk
        pairwise = current_embeddings @ current_embeddings.T

        min_sim = float(
            np.min(
                pairwise[
                    np.triu_indices(len(current), k=1)
                ]
            )
        )

        threshold = max(
            c * min_sim * sigmoid(len(current) - 1),
            hard_thr
        )

        if max_sim > threshold:
            current.append(i)
        else:
            chunks.append(tuple(current))
            current = [i]

    chunks.append(tuple(current))

    return chunks


print("=" * 60)
print("MAX–MIN IMPLEMENTATION READY")
print("=" * 60)

print("hard_thr   =", 0.6)
print("c          =", 0.9)
print("init_const =", 1.5)
print("Similarity =", "dot product of L2-normalized embeddings")
print("Window     =", "none")
print("Max size   =", "none")

print("\nFunction ready.")

MAX–MIN IMPLEMENTATION READY
hard_thr   = 0.6
c          = 0.9
init_const = 1.5
Similarity = dot product of L2-normalized embeddings
Window     = none
Max size   = none

Function ready.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 13
# Verify Checkpoint Resume State
# ============================================================

MANIFEST_PATH = CKPT_DIR / "manifest.json"

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

completed = set(manifest["completed_docids"])

print("=" * 60)
print("MAX–MIN RESUME STATE")
print("=" * 60)

print("Corpus documents:", len(corpus))
print("Checkpointed documents:", len(completed))
print("Remaining documents:", len(set(corpus) - completed))

# Confirm every checkpointed doc still exists in the current corpus
missing_from_corpus = completed - set(corpus)

# Confirm no unexpected checkpointed IDs
unexpected = completed - set(docids)

print("\nCheckpointed docs missing from corpus:", len(missing_from_corpus))
print("Unexpected checkpoint IDs:", len(unexpected))

# Show the next documents in the current corpus order
remaining_docids = [d for d in docids if d not in completed]

print("\nFirst 10 remaining documents:")
for docid in remaining_docids[:10]:
    print(" ", docid)

print("\nResume should begin with:", remaining_docids[0])

MAX–MIN RESUME STATE
Corpus documents: 3806
Checkpointed documents: 1900
Remaining documents: 1906

Checkpointed docs missing from corpus: 0
Unexpected checkpoint IDs: 0

First 10 remaining documents:
  doc-hi-328
  doc-hi-2557
  doc-hi-6092
  doc-hi-3692
  doc-hi-1358
  doc-hi-2291
  doc-hi-2431
  doc-hi-6374
  doc-hi-744
  doc-hi-3862

Resume should begin with: doc-hi-328


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 14
# Resume Max–Min Segmentation
# ============================================================

from sentence_transformers import SentenceTransformer

print("=" * 60)
print("MLDR 003 — RESUMING MAX–MIN")
print("=" * 60)

# Clear anything left on GPU
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("GPU memory cleared.")

print("\nLoading BGE-M3...")

sentence_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda"
)

sentence_model.max_seq_length = 8192

print("Model loaded.")
print("Max sequence length:", sentence_model.max_seq_length)

# Rebuild resume list
remaining_docids = [
    d for d in docids
    if d not in completed
]

print("\nTotal corpus:", len(docids))
print("Already completed:", len(completed))
print("Remaining:", len(remaining_docids))

EMBED_BATCH_SIZE = 8
CHECKPOINT_DOCS = 25

print("\nEmbedding batch size:", EMBED_BATCH_SIZE)
print("Checkpoint frequency:", CHECKPOINT_DOCS, "documents")

total_start = time.perf_counter()

for start in range(0, len(remaining_docids), CHECKPOINT_DOCS):

    batch_docids = remaining_docids[
        start:start + CHECKPOINT_DOCS
    ]

    print("\n" + "=" * 60)
    print(
        f"RESUME BATCH "
        f"{len(completed) + 1}-"
        f"{len(completed) + len(batch_docids)}"
    )
    print("=" * 60)

    batch_results = {}
    batch_start = time.perf_counter()

    for local_idx, docid in enumerate(batch_docids, start=1):

        text = corpus[docid]
        sentences = split_sentences(text)

        if not sentences:
            batch_results[docid] = {
                "sentences": [],
                "embeddings": np.empty(
                    (0, 1024),
                    dtype=np.float32
                ),
                "chunks": []
            }
            continue

        embeddings = sentence_model.encode(
            sentences,
            batch_size=EMBED_BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True
        ).astype(np.float32)

        chunks = maxmin_segment(
            sentences,
            embeddings
        )

        batch_results[docid] = {
            "sentences": sentences,
            "embeddings": embeddings,
            "chunks": [list(chunk) for chunk in chunks]
        }

        del embeddings

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if local_idx % 5 == 0:
            elapsed = time.perf_counter() - batch_start
            print(
                f"  {local_idx}/{len(batch_docids)} docs "
                f"| {elapsed:.1f}s"
            )

    end_number = (
        len(completed) +
        len(batch_docids)
    )

    checkpoint_path = (
        CKPT_DIR /
        f"batch_{end_number:04d}.pkl"
    )

    with open(checkpoint_path, "wb") as f:
        pickle.dump(
            batch_results,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    completed.update(batch_docids)
    manifest["completed_docids"] = sorted(completed)

    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(
            manifest,
            f,
            ensure_ascii=False,
            indent=2
        )

    batch_elapsed = (
        time.perf_counter() -
        batch_start
    )

    total_elapsed = (
        time.perf_counter() -
        total_start
    )

    print(
        f"\nCHECKPOINT SAVED: "
        f"{checkpoint_path.name}"
    )
    print(
        f"Completed: "
        f"{len(completed)}/{len(docids)}"
    )
    print(
        f"Batch time: "
        f"{batch_elapsed / 60:.2f} min"
    )
    print(
        f"Session time: "
        f"{total_elapsed / 60:.2f} min"
    )

    del batch_results

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print("\n" + "=" * 60)
print("RESUME COMPLETE")
print("=" * 60)

print(
    "Checkpointed documents:",
    len(completed)
)

print(
    "Expected:",
    len(docids)
)

MLDR 003 — RESUMING MAX–MIN
GPU memory cleared.

Loading BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded.
Max sequence length: 8192

Total corpus: 3806
Already completed: 1900
Remaining: 1906

Embedding batch size: 8
Checkpoint frequency: 25 documents

RESUME BATCH 1901-1925
  5/25 docs | 9.3s
  10/25 docs | 14.3s
  15/25 docs | 20.0s
  20/25 docs | 25.2s
  25/25 docs | 32.1s

CHECKPOINT SAVED: batch_1925.pkl
Completed: 1925/3806
Batch time: 0.54 min
Session time: 0.54 min

RESUME BATCH 1926-1950
  5/25 docs | 21.4s
  10/25 docs | 58.4s
  15/25 docs | 66.1s
  20/25 docs | 78.2s
  25/25 docs | 91.0s

CHECKPOINT SAVED: batch_1950.pkl
Completed: 1950/3806
Batch time: 1.52 min
Session time: 2.06 min

RESUME BATCH 1951-1975
  5/25 docs | 8.7s
  10/25 docs | 18.9s
  15/25 docs | 25.7s
  20/25 docs | 33.5s
  25/25 docs | 38.3s

CHECKPOINT SAVED: batch_1975.pkl
Completed: 1975/3806
Batch time: 0.64 min
Session time: 2.70 min

RESUME BATCH 1976-2000
  5/25 docs | 5.0s
  10/25 docs | 11.7s
  15/25 docs | 17.6s
  20/25 docs | 26.3s
  25/25 docs | 31.0s

CHECKPOINT SAVED: batch_2000.pkl


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 15
# Reconstruct Max–Min + Structural Audit
# ============================================================

print("=" * 60)
print("RECONSTRUCTING MLDR 003 MAX–MIN")
print("=" * 60)

checkpoint_files = sorted(CKPT_DIR.glob("batch_*.pkl"))

print("Checkpoint files:", len(checkpoint_files))

maxmin_segments = {}
total_chunks = 0
total_sentences = 0
singleton_chunks = 0
chunk_sentence_lengths = []

for i, checkpoint_path in enumerate(checkpoint_files, start=1):

    with open(checkpoint_path, "rb") as f:
        batch = pickle.load(f)

    for docid, result in batch.items():

        chunks = result["chunks"]

        maxmin_segments[docid] = chunks

        total_chunks += len(chunks)
        total_sentences += len(result["sentences"])

        for chunk in chunks:
            n = len(chunk)
            chunk_sentence_lengths.append(n)

            if n == 1:
                singleton_chunks += 1

    if i % 10 == 0 or i == len(checkpoint_files):
        print(
            f"  Loaded {i}/{len(checkpoint_files)} checkpoints"
        )

chunk_sentence_lengths = np.array(
    chunk_sentence_lengths,
    dtype=np.int32
)

print("\n" + "=" * 60)
print("STRUCTURAL SUMMARY")
print("=" * 60)

print("Documents:", len(maxmin_segments))
print("Total sentences:", total_sentences)
print("Total chunks:", total_chunks)

print(
    "Mean chunks/document:",
    total_chunks / len(maxmin_segments)
)

print(
    "Mean sentences/chunk:",
    np.mean(chunk_sentence_lengths)
)

print(
    "Median sentences/chunk:",
    np.median(chunk_sentence_lengths)
)

print(
    "Singleton chunks:",
    singleton_chunks,
    f"({100 * singleton_chunks / total_chunks:.2f}%)"
)

print("\nChunk sentence-length distribution:")

for p in [50, 75, 90, 95, 99, 99.5, 100]:
    print(
        f"  P{p:<4}: "
        f"{np.percentile(chunk_sentence_lengths, p):.2f}"
    )

print("\nMax sentences in a chunk:",
      np.max(chunk_sentence_lengths))

print("\nReconstruction complete.")

RECONSTRUCTING MLDR 003 MAX–MIN
Checkpoint files: 96
  Loaded 10/96 checkpoints
  Loaded 20/96 checkpoints
  Loaded 30/96 checkpoints
  Loaded 40/96 checkpoints
  Loaded 50/96 checkpoints
  Loaded 60/96 checkpoints
  Loaded 70/96 checkpoints
  Loaded 80/96 checkpoints
  Loaded 90/96 checkpoints
  Loaded 96/96 checkpoints

STRUCTURAL SUMMARY
Documents: 3806
Total sentences: 540259
Total chunks: 238301
Mean chunks/document: 62.61192853389385
Mean sentences/chunk: 2.2671285475092424
Median sentences/chunk: 2.0
Singleton chunks: 51098 (21.44%)

Chunk sentence-length distribution:
  P50  : 2.00
  P75  : 2.00
  P90  : 3.00
  P95  : 5.00
  P99  : 9.00
  P99.5: 11.00
  P100 : 1305.00

Max sentences in a chunk: 1305

Reconstruction complete.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 16
# Max–Min Chunk Token-Length Audit
# ============================================================

print("=" * 60)
print("MAX–MIN CHUNK TOKEN-LENGTH AUDIT")
print("=" * 60)

# Reuse the BGE-M3 tokenizer already loaded
chunk_token_lengths = []
oversized_examples = []

for i, checkpoint_path in enumerate(checkpoint_files, start=1):

    with open(checkpoint_path, "rb") as f:
        batch = pickle.load(f)

    for docid, result in batch.items():

        sentences = result["sentences"]
        chunks = result["chunks"]

        for chunk_idx, chunk in enumerate(chunks):

            chunk_text = " ".join(
                sentences[sidx] for sidx in chunk
            )

            n_tokens = len(
                tokenizer(
                    chunk_text,
                    add_special_tokens=True,
                    truncation=False
                )["input_ids"]
            )

            chunk_token_lengths.append(n_tokens)

            if n_tokens > 8192 and len(oversized_examples) < 20:
                oversized_examples.append(
                    (docid, chunk_idx, len(chunk), n_tokens)
                )

    if i % 10 == 0 or i == len(checkpoint_files):
        print(
            f"  Processed {i}/{len(checkpoint_files)} checkpoints"
        )

chunk_token_lengths = np.asarray(
    chunk_token_lengths,
    dtype=np.int32
)

print("\n" + "=" * 60)
print("TOKEN-LENGTH SUMMARY")
print("=" * 60)

print("Total chunks:", len(chunk_token_lengths))

for p in [50, 75, 90, 95, 99, 99.5, 100]:
    print(
        f"P{p:<4}: "
        f"{np.percentile(chunk_token_lengths, p):.2f}"
    )

print("\nThreshold counts:")

for threshold in [256, 512, 1024, 2048, 4096, 8192]:
    count = np.sum(chunk_token_lengths > threshold)
    print(
        f"> {threshold:5d}: "
        f"{count:6d} "
        f"({100 * count / len(chunk_token_lengths):.4f}%)"
    )

print("\nMaximum chunk tokens:",
      int(np.max(chunk_token_lengths)))

print("\nExamples of chunks >8192 tokens:")
for row in oversized_examples:
    print(
        f"  doc={row[0]} | "
        f"chunk={row[1]} | "
        f"sentences={row[2]} | "
        f"tokens={row[3]}"
    )

print("\nAudit complete.")

MAX–MIN CHUNK TOKEN-LENGTH AUDIT
  Processed 10/96 checkpoints
  Processed 20/96 checkpoints
  Processed 30/96 checkpoints
  Processed 40/96 checkpoints
  Processed 50/96 checkpoints
  Processed 60/96 checkpoints
  Processed 70/96 checkpoints
  Processed 80/96 checkpoints
  Processed 90/96 checkpoints
  Processed 96/96 checkpoints

TOKEN-LENGTH SUMMARY
Total chunks: 238301
P50  : 67.00
P75  : 105.00
P90  : 164.00
P95  : 225.00
P99  : 462.00
P99.5: 616.00
P100 : 92329.00

Threshold counts:
>   256:   8933 (3.7486%)
>   512:   1850 (0.7763%)
>  1024:    386 (0.1620%)
>  2048:    106 (0.0445%)
>  4096:     45 (0.0189%)
>  8192:     18 (0.0076%)

Maximum chunk tokens: 92329

Examples of chunks >8192 tokens:
  doc=doc-hi-6356 | chunk=0 | sentences=3 | tokens=16276
  doc=doc-hi-1355 | chunk=11 | sentences=5 | tokens=26608
  doc=doc-hi-306 | chunk=0 | sentences=4 | tokens=15463
  doc=doc-hi-5111 | chunk=1 | sentences=641 | tokens=92329
  doc=doc-hi-6613 | chunk=26 | sentences=1305 | tokens=28

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 17
# Build Final Max–Min Chunk Corpus
# ============================================================

print("=" * 60)
print("BUILDING FINAL MAX–MIN CHUNK CORPUS")
print("=" * 60)

MAX_TOKENS = 8192

maxmin_chunk_records = []
oversized_chunk_records = []

for i, checkpoint_path in enumerate(checkpoint_files, start=1):

    with open(checkpoint_path, "rb") as f:
        batch = pickle.load(f)

    for docid, result in batch.items():

        sentences = result["sentences"]
        chunks = result["chunks"]

        for chunk_idx, sentence_indices in enumerate(chunks):

            chunk_text = " ".join(
                sentences[sidx] for sidx in sentence_indices
            )

            token_count = len(
                tokenizer(
                    chunk_text,
                    add_special_tokens=True,
                    truncation=False
                )["input_ids"]
            )

            record = {
                "doc_id": docid,
                "chunk_idx": chunk_idx,
                "sentence_indices": sentence_indices,
                "text": chunk_text,
                "token_count": token_count,
            }

            maxmin_chunk_records.append(record)

            if token_count > MAX_TOKENS:
                oversized_chunk_records.append(record)

    if i % 10 == 0 or i == len(checkpoint_files):
        print(
            f"  Processed {i}/{len(checkpoint_files)} checkpoints"
        )

print("\n" + "=" * 60)
print("FINAL CORPUS")
print("=" * 60)

print("Documents:", len(maxmin_segments))
print("Chunks:", len(maxmin_chunk_records))

print(
    "Oversized chunks (>8192):",
    len(oversized_chunk_records),
    f"({100 * len(oversized_chunk_records) / len(maxmin_chunk_records):.4f}%)"
)

print(
    "Maximum original chunk tokens:",
    max(r["token_count"] for r in maxmin_chunk_records)
)

print("\nOversized chunks:")
for r in oversized_chunk_records:
    print(
        f"  {r['doc_id']} | "
        f"chunk={r['chunk_idx']} | "
        f"sentences={len(r['sentence_indices'])} | "
        f"tokens={r['token_count']}"
    )

print("\nCorpus construction complete.")

BUILDING FINAL MAX–MIN CHUNK CORPUS
  Processed 10/96 checkpoints
  Processed 20/96 checkpoints
  Processed 30/96 checkpoints
  Processed 40/96 checkpoints
  Processed 50/96 checkpoints
  Processed 60/96 checkpoints
  Processed 70/96 checkpoints
  Processed 80/96 checkpoints
  Processed 90/96 checkpoints
  Processed 96/96 checkpoints

FINAL CORPUS
Documents: 3806
Chunks: 238301
Oversized chunks (>8192): 18 (0.0076%)
Maximum original chunk tokens: 92329

Oversized chunks:
  doc-hi-6356 | chunk=0 | sentences=3 | tokens=16276
  doc-hi-1355 | chunk=11 | sentences=5 | tokens=26608
  doc-hi-306 | chunk=0 | sentences=4 | tokens=15463
  doc-hi-5111 | chunk=1 | sentences=641 | tokens=92329
  doc-hi-6613 | chunk=26 | sentences=1305 | tokens=28280
  doc-hi-6684 | chunk=22 | sentences=147 | tokens=8536
  doc-hi-6698 | chunk=21 | sentences=5 | tokens=9477
  doc-hi-6740 | chunk=1 | sentences=1 | tokens=16990
  doc-hi-6741 | chunk=0 | sentences=2 | tokens=12723
  doc-hi-6742 | chunk=0 | sentences=3 |

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 18
# Save Frozen Max–Min Chunk Corpus
# ============================================================

import json

FROZEN_DIR = RESEARCH / "frozen"
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = FROZEN_DIR / "mldr003_maxmin_chunk_corpus.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        maxmin_chunk_records,
        f,
        ensure_ascii=False,
        separators=(",", ":")
    )

print("=" * 60)
print("FROZEN CORPUS SAVED")
print("=" * 60)

print("Path:", OUT_PATH)
print("Chunks:", len(maxmin_chunk_records))
print(
    "Oversized chunks:",
    len(oversized_chunk_records)
)

print(
    "File size:",
    f"{OUT_PATH.stat().st_size / (1024**2):.2f} MB"
)

print("Status: OK")

FROZEN CORPUS SAVED
Path: /content/drive/MyDrive/maxmin_research/frozen/mldr003_maxmin_chunk_corpus.json
Chunks: 238301
Oversized chunks: 18
File size: 189.86 MB
Status: OK


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 19
# BGE-M3 Embedding Setup
# ============================================================

EMB_DIR = RESEARCH / "cache" / "mldr003_embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "BAAI/bge-m3"
MAX_MODEL_TOKENS = 8192
EMBED_BATCH_SIZE = 8

print("=" * 60)
print("BGE-M3 EMBEDDING SETUP")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Max tokens:", MAX_MODEL_TOKENS)
print("Embedding batch size:", EMBED_BATCH_SIZE)
print("Cache:", EMB_DIR)

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

# Verify tokenizer configuration
test = tokenizer(
    "यह एक परीक्षण वाक्य है।",
    truncation=True,
    max_length=MAX_MODEL_TOKENS
)

print("Tokenizer test tokens:", len(test["input_ids"]))

print("\nSetup OK.")

BGE-M3 EMBEDDING SETUP
Model: BAAI/bge-m3
Max tokens: 8192
Embedding batch size: 8
Cache: /content/drive/MyDrive/maxmin_research/cache/mldr003_embeddings

GPU: Tesla T4
CUDA: True
Tokenizer test tokens: 8

Setup OK.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 20
# Resumable BGE-M3 Max–Min Chunk Embedding
# ============================================================

import gc
import time
import pickle

EMB_BATCH = 8
EMB_CHECKPOINT = 2000

N_CHUNKS = len(maxmin_chunk_records)
EMB_DIM = 1024

EMB_MATRIX_PATH = EMB_DIR / "maxmin_chunk_embeddings.npy"
EMB_STATE_PATH = EMB_DIR / "maxmin_embedding_state.pkl"

# ------------------------------------------------------------
# Load / initialize embedding matrix
# ------------------------------------------------------------

if EMB_MATRIX_PATH.exists() and EMB_STATE_PATH.exists():

    embeddings = np.load(EMB_MATRIX_PATH, mmap_mode="r+")

    with open(EMB_STATE_PATH, "rb") as f:
        state = pickle.load(f)

    start_idx = state["completed"]

    print("Resuming existing embedding run.")
    print("Completed:", start_idx, "/", N_CHUNKS)

else:

    embeddings = np.lib.format.open_memmap(
        EMB_MATRIX_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(N_CHUNKS, EMB_DIM)
    )

    start_idx = 0

    with open(EMB_STATE_PATH, "wb") as f:
        pickle.dump(
            {
                "completed": 0,
                "total": N_CHUNKS,
                "model": MODEL_NAME,
                "max_tokens": MAX_MODEL_TOKENS,
                "batch_size": EMB_BATCH,
            },
            f
        )

    print("Starting new embedding run.")
    print("Total chunks:", N_CHUNKS)

# ------------------------------------------------------------
# Embedding loop
# ------------------------------------------------------------

model.eval()

run_start = time.time()

for start in range(start_idx, N_CHUNKS, EMB_BATCH):

    end = min(start + EMB_BATCH, N_CHUNKS)

    texts = [
        r["text"]
        for r in maxmin_chunk_records[start:end]
    ]

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_MODEL_TOKENS,
        return_tensors="pt"
    )

    encoded = {
        k: v.to("cuda")
        for k, v in encoded.items()
    }

    with torch.no_grad():
        output = model(**encoded)

        batch_emb = output.last_hidden_state[:, 0, :]
        batch_emb = torch.nn.functional.normalize(
            batch_emb,
            p=2,
            dim=1
        )

    embeddings[start:end] = (
        batch_emb
        .float()
        .cpu()
        .numpy()
    )

    del encoded, output, batch_emb

    if end % EMB_CHECKPOINT == 0 or end == N_CHUNKS:

        embeddings.flush()

        with open(EMB_STATE_PATH, "wb") as f:
            pickle.dump(
                {
                    "completed": end,
                    "total": N_CHUNKS,
                    "model": MODEL_NAME,
                    "max_tokens": MAX_MODEL_TOKENS,
                    "batch_size": EMB_BATCH,
                },
                f
            )

        elapsed = time.time() - run_start
        rate = (end - start_idx) / max(elapsed, 1e-9)

        print(
            f"Checkpoint: {end:,}/{N_CHUNKS:,} "
            f"({100*end/N_CHUNKS:.2f}%)"
        )

    if start % 256 == 0:
        gc.collect()
        torch.cuda.empty_cache()

embeddings.flush()

print("\n" + "=" * 60)
print("EMBEDDING COMPLETE")
print("=" * 60)

print("Embeddings:", embeddings.shape)
print("dtype:", embeddings.dtype)
print("Path:", EMB_MATRIX_PATH)
print("Completed:", N_CHUNKS)

Resuming existing embedding run.
Completed: 0 / 238301
Checkpoint: 2,000/238,301 (0.84%)
Checkpoint: 4,000/238,301 (1.68%)
Checkpoint: 6,000/238,301 (2.52%)
Checkpoint: 8,000/238,301 (3.36%)
Checkpoint: 10,000/238,301 (4.20%)
Checkpoint: 12,000/238,301 (5.04%)
Checkpoint: 14,000/238,301 (5.87%)
Checkpoint: 16,000/238,301 (6.71%)
Checkpoint: 18,000/238,301 (7.55%)
Checkpoint: 20,000/238,301 (8.39%)
Checkpoint: 22,000/238,301 (9.23%)
Checkpoint: 24,000/238,301 (10.07%)
Checkpoint: 26,000/238,301 (10.91%)
Checkpoint: 28,000/238,301 (11.75%)
Checkpoint: 30,000/238,301 (12.59%)
Checkpoint: 32,000/238,301 (13.43%)
Checkpoint: 34,000/238,301 (14.27%)
Checkpoint: 36,000/238,301 (15.11%)
Checkpoint: 38,000/238,301 (15.95%)
Checkpoint: 40,000/238,301 (16.79%)
Checkpoint: 42,000/238,301 (17.62%)
Checkpoint: 44,000/238,301 (18.46%)
Checkpoint: 46,000/238,301 (19.30%)
Checkpoint: 48,000/238,301 (20.14%)
Checkpoint: 50,000/238,301 (20.98%)
Checkpoint: 52,000/238,301 (21.82%)
Checkpoint: 54,000/238,3

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 20A
# Load BGE-M3
# ============================================================

from transformers import AutoModel

print("Loading:", MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

model.eval()

print("=" * 60)
print("MODEL LOADED")
print("=" * 60)
print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("dtype:", next(model.parameters()).dtype)

Loading: BAAI/bge-m3


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

MODEL LOADED
Model: BAAI/bge-m3
Device: cuda:0
dtype: torch.float16


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 21
# Reconnect + Inspect Embedding Progress
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import pickle
import numpy as np

BASE = Path("/content/drive/MyDrive")
RESEARCH = BASE / "maxmin_research"

EMB_DIR = RESEARCH / "cache" / "mldr003_embeddings"
EMB_STATE_PATH = EMB_DIR / "maxmin_embedding_state.pkl"
EMB_MATRIX_PATH = EMB_DIR / "maxmin_chunk_embeddings.npy"

print("=" * 60)
print("CHECKING SAVED EMBEDDING STATE")
print("=" * 60)

print("Embedding directory exists:", EMB_DIR.exists())
print("State file exists:", EMB_STATE_PATH.exists())
print("Embedding matrix exists:", EMB_MATRIX_PATH.exists())

if EMB_STATE_PATH.exists():

    with open(EMB_STATE_PATH, "rb") as f:
        state = pickle.load(f)

    print("\nSaved state:")
    print("  Completed:", state["completed"])
    print("  Total:", state["total"])
    print(
        "  Progress:",
        f"{100 * state['completed'] / state['total']:.2f}%"
    )
    print("  Model:", state["model"])
    print("  Batch size:", state["batch_size"])

if EMB_MATRIX_PATH.exists():
    print(
        "\nEmbedding matrix size:",
        f"{EMB_MATRIX_PATH.stat().st_size / (1024**3):.2f} GB"
    )

Mounted at /content/drive
CHECKING SAVED EMBEDDING STATE
Embedding directory exists: True
State file exists: True
Embedding matrix exists: True

Saved state:
  Completed: 160000
  Total: 238301
  Progress: 67.14%
  Model: BAAI/bge-m3
  Batch size: 8

Embedding matrix size: 0.91 GB


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 22
# Restore BGE-M3 After Runtime Disconnect
# ============================================================

import torch
import gc
import time
import pickle
import numpy as np

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "BAAI/bge-m3"
MAX_MODEL_TOKENS = 8192
EMB_BATCH = 8
EMB_CHECKPOINT = 2000

EMB_DIR = RESEARCH / "cache" / "mldr003_embeddings"
EMB_MATRIX_PATH = EMB_DIR / "maxmin_chunk_embeddings.npy"
EMB_STATE_PATH = EMB_DIR / "maxmin_embedding_state.pkl"

print("=" * 60)
print("RESTORING BGE-M3")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
).to("cuda")

model.eval()

print("\nModel:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("dtype:", next(model.parameters()).dtype)
print("Status: READY")

RESTORING BGE-M3
GPU: Tesla T4
CUDA: True


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Model: BAAI/bge-m3
Device: cuda:0
dtype: torch.float16
Status: READY


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 23
# Fully Self-Contained Resume
# ============================================================

import gc
import time
import pickle
import numpy as np
import torch

EMB_BATCH = 8
EMB_CHECKPOINT = 2000
MAX_MODEL_TOKENS = 8192
MODEL_NAME = "BAAI/bge-m3"

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")

EMB_DIR = RESEARCH / "cache" / "mldr003_embeddings"
EMB_MATRIX_PATH = EMB_DIR / "maxmin_chunk_embeddings.npy"
EMB_STATE_PATH = EMB_DIR / "maxmin_embedding_state.pkl"

# ------------------------------------------------------------
# Load frozen corpus
# ------------------------------------------------------------

CORPUS_PATH = (
    RESEARCH /
    "frozen" /
    "mldr003_maxmin_chunk_corpus.json"
)

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    maxmin_chunk_records = json.load(f)

N_CHUNKS = len(maxmin_chunk_records)

# ------------------------------------------------------------
# Load saved state
# ------------------------------------------------------------

with open(EMB_STATE_PATH, "rb") as f:
    state = pickle.load(f)

start_idx = state["completed"]

print("=" * 60)
print("RESUMING EMBEDDING RUN")
print("=" * 60)

print(f"Starting from: {start_idx:,}/{N_CHUNKS:,}")
print(f"Remaining: {N_CHUNKS - start_idx:,}")
print(f"Progress: {100 * start_idx / N_CHUNKS:.2f}%")

# ------------------------------------------------------------
# Open existing Drive-backed matrix
# ------------------------------------------------------------

embeddings = np.load(
    EMB_MATRIX_PATH,
    mmap_mode="r+"
)

assert embeddings.shape == (N_CHUNKS, 1024)

print("Embedding matrix:", embeddings.shape)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Resume
# ------------------------------------------------------------

model.eval()

run_start = time.time()

for start in range(start_idx, N_CHUNKS, EMB_BATCH):

    end = min(start + EMB_BATCH, N_CHUNKS)

    texts = [
        r["text"]
        for r in maxmin_chunk_records[start:end]
    ]

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_MODEL_TOKENS,
        return_tensors="pt"
    )

    encoded = {
        k: v.to("cuda")
        for k, v in encoded.items()
    }

    with torch.no_grad():
        output = model(**encoded)

        batch_emb = output.last_hidden_state[:, 0, :]
        batch_emb = torch.nn.functional.normalize(
            batch_emb,
            p=2,
            dim=1
        )

    embeddings[start:end] = (
        batch_emb
        .float()
        .cpu()
        .numpy()
    )

    del encoded, output, batch_emb

    if end % EMB_CHECKPOINT == 0 or end == N_CHUNKS:

        embeddings.flush()

        with open(EMB_STATE_PATH, "wb") as f:
            pickle.dump(
                {
                    "completed": end,
                    "total": N_CHUNKS,
                    "model": MODEL_NAME,
                    "max_tokens": MAX_MODEL_TOKENS,
                    "batch_size": EMB_BATCH,
                },
                f
            )

        elapsed = time.time() - run_start

        print(
            f"Checkpoint: {end:,}/{N_CHUNKS:,} "
            f"({100 * end / N_CHUNKS:.2f}%)"
        )

    if start % 256 == 0:
        gc.collect()
        torch.cuda.empty_cache()

embeddings.flush()

print("\n" + "=" * 60)
print("EMBEDDING COMPLETE")
print("=" * 60)
print("Shape:", embeddings.shape)
print("Completed:", N_CHUNKS)

RESUMING EMBEDDING RUN
Starting from: 160,000/238,301
Remaining: 78,301
Progress: 67.14%
Embedding matrix: (238301, 1024)
GPU: Tesla T4
Checkpoint: 162,000/238,301 (67.98%)
Checkpoint: 164,000/238,301 (68.82%)
Checkpoint: 166,000/238,301 (69.66%)
Checkpoint: 168,000/238,301 (70.50%)
Checkpoint: 170,000/238,301 (71.34%)
Checkpoint: 172,000/238,301 (72.18%)
Checkpoint: 174,000/238,301 (73.02%)
Checkpoint: 176,000/238,301 (73.86%)
Checkpoint: 178,000/238,301 (74.70%)
Checkpoint: 180,000/238,301 (75.53%)
Checkpoint: 182,000/238,301 (76.37%)
Checkpoint: 184,000/238,301 (77.21%)
Checkpoint: 186,000/238,301 (78.05%)
Checkpoint: 188,000/238,301 (78.89%)
Checkpoint: 190,000/238,301 (79.73%)
Checkpoint: 192,000/238,301 (80.57%)
Checkpoint: 194,000/238,301 (81.41%)
Checkpoint: 196,000/238,301 (82.25%)
Checkpoint: 198,000/238,301 (83.09%)
Checkpoint: 200,000/238,301 (83.93%)
Checkpoint: 202,000/238,301 (84.77%)
Checkpoint: 204,000/238,301 (85.61%)
Checkpoint: 206,000/238,301 (86.45%)
Checkpoint: 2

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 24
# Max–Min Embedding Integrity Check
# ============================================================

import numpy as np

EMB_MATRIX_PATH = (
    Path("/content/drive/MyDrive/maxmin_research")
    / "cache"
    / "mldr003_embeddings"
    / "maxmin_chunk_embeddings.npy"
)

embeddings = np.load(
    EMB_MATRIX_PATH,
    mmap_mode="r"
)

print("=" * 60)
print("MAX–MIN EMBEDDING INTEGRITY")
print("=" * 60)

print("Shape:", embeddings.shape)
print("dtype:", embeddings.dtype)

# ------------------------------------------------------------
# 1. NaN / Inf check
# ------------------------------------------------------------

nan_count = np.isnan(embeddings).sum()
inf_count = np.isinf(embeddings).sum()

print("\nNaN values:", nan_count)
print("Inf values:", inf_count)

# ------------------------------------------------------------
# 2. L2 norm check
# ------------------------------------------------------------

sample_idx = np.linspace(
    0,
    len(embeddings) - 1,
    1000,
    dtype=np.int64
)

sample = np.asarray(
    embeddings[sample_idx],
    dtype=np.float32
)

norms = np.linalg.norm(sample, axis=1)

print("\nL2 norm check — 1000 samples:")
print("  Min:", norms.min())
print("  Mean:", norms.mean())
print("  Max:", norms.max())
print(
    "  Max |norm - 1|:",
    np.max(np.abs(norms - 1.0))
)

# ------------------------------------------------------------
# 3. Show a few embedding values
# ------------------------------------------------------------

print("\nFirst 5 dimensions of 3 embeddings:")

for idx in [0, len(embeddings)//2, len(embeddings)-1]:
    print(
        f"  [{idx}]:",
        embeddings[idx][:5]
    )

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

assert embeddings.shape == (238301, 1024)
assert nan_count == 0
assert inf_count == 0
assert np.max(np.abs(norms - 1.0)) < 1e-3

print("\n" + "=" * 60)
print("INTEGRITY CHECK: PASS")
print("=" * 60)

MAX–MIN EMBEDDING INTEGRITY
Shape: (238301, 1024)
dtype: float32

NaN values: 0
Inf values: 0

L2 norm check — 1000 samples:
  Min: 0.0
  Mean: 0.3289954
  Max: 1.0003313
  Max |norm - 1|: 1.0

First 5 dimensions of 3 embeddings:
  [0]: [0. 0. 0. 0. 0.]
  [119150]: [0. 0. 0. 0. 0.]
  [238300]: [ 0.03805542  0.00580597 -0.04653931  0.04638672 -0.01913452]


AssertionError: 

In [ ]:
import json
from pathlib import Path

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 25
# Locate Zero Embeddings
# ============================================================

import numpy as np
from pathlib import Path
import pickle

EMB_DIR = (
    Path("/content/drive/MyDrive/maxmin_research")
    / "cache"
    / "mldr003_embeddings"
)

EMB_MATRIX_PATH = EMB_DIR / "maxmin_chunk_embeddings.npy"
EMB_STATE_PATH = EMB_DIR / "maxmin_embedding_state.pkl"

embeddings = np.load(
    EMB_MATRIX_PATH,
    mmap_mode="r"
)

N = embeddings.shape[0]

print("=" * 60)
print("ZERO-VECTOR AUDIT")
print("=" * 60)

# Check every row in manageable blocks
zero_indices = []
nonzero_count = 0

BLOCK = 10000

for start in range(0, N, BLOCK):

    end = min(start + BLOCK, N)

    block = np.asarray(
        embeddings[start:end],
        dtype=np.float32
    )

    norms = np.linalg.norm(block, axis=1)

    zeros = np.where(norms == 0)[0]

    if len(zeros):
        zero_indices.extend(
            (zeros + start).tolist()
        )

    nonzero_count += np.count_nonzero(norms)

print("Total rows:", N)
print("Non-zero rows:", nonzero_count)
print("Zero rows:", len(zero_indices))

if zero_indices:
    print(
        "Zero percentage:",
        f"{100 * len(zero_indices) / N:.4f}%"
    )

    print("\nFirst 20 zero indices:")
    print(zero_indices[:20])

    print("\nLast 20 zero indices:")
    print(zero_indices[-20:])

    # Find contiguous zero ranges
    ranges = []

    start = prev = zero_indices[0]

    for idx in zero_indices[1:]:
        if idx == prev + 1:
            prev = idx
        else:
            ranges.append((start, prev))
            start = prev = idx

    ranges.append((start, prev))

    print("\nContiguous zero ranges:")
    for a, b in ranges[:30]:
        print(
            f"  {a:,} - {b:,} "
            f"({b-a+1:,} rows)"
        )

print("\nSaved state:")

with open(EMB_STATE_PATH, "rb") as f:
    state = pickle.load(f)

print(state)

print("\nAudit complete.")

ZERO-VECTOR AUDIT
Total rows: 238301
Non-zero rows: 78301
Zero rows: 160000
Zero percentage: 67.1420%

First 20 zero indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Last 20 zero indices:
[159980, 159981, 159982, 159983, 159984, 159985, 159986, 159987, 159988, 159989, 159990, 159991, 159992, 159993, 159994, 159995, 159996, 159997, 159998, 159999]

Contiguous zero ranges:
  0 - 159,999 (160,000 rows)

Saved state:
{'completed': 238301, 'total': 238301, 'model': 'BAAI/bge-m3', 'max_tokens': 8192, 'batch_size': 8}

Audit complete.


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 26
# Repair Missing First 160,000 Embeddings
# ============================================================

import gc
import time
import pickle
import json
import numpy as np
import torch

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")

CORPUS_PATH = (
    RESEARCH /
    "frozen" /
    "mldr003_maxmin_chunk_corpus.json"
)

EMB_DIR = (
    RESEARCH /
    "cache" /
    "mldr003_embeddings"
)

EMB_MATRIX_PATH = EMB_DIR / "maxmin_chunk_embeddings.npy"
EMB_STATE_PATH = EMB_DIR / "maxmin_embedding_state.pkl"

# ------------------------------------------------------------
# Load frozen corpus
# ------------------------------------------------------------

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    maxmin_chunk_records = json.load(f)

N_CHUNKS = len(maxmin_chunk_records)
REPAIR_END = 160000

print("=" * 60)
print("REPAIRING MISSING EMBEDDINGS")
print("=" * 60)

print("Total chunks:", N_CHUNKS)
print("Repair range: 0 ->", REPAIR_END)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Open existing matrix
# ------------------------------------------------------------

embeddings = np.load(
    EMB_MATRIX_PATH,
    mmap_mode="r+"
)

assert embeddings.shape == (N_CHUNKS, 1024)

# Confirm target region is actually zero
probe = np.asarray(
    embeddings[:1000],
    dtype=np.float32
)

zero_probe = np.sum(
    np.linalg.norm(probe, axis=1) == 0
)

print("Zero rows in first 1,000:", zero_probe)

assert zero_probe == 1000

# ------------------------------------------------------------
# Repair
# ------------------------------------------------------------

EMB_BATCH = 8
CHECKPOINT = 2000

model.eval()

run_start = time.time()

for start in range(0, REPAIR_END, EMB_BATCH):

    end = min(start + EMB_BATCH, REPAIR_END)

    texts = [
        r["text"]
        for r in maxmin_chunk_records[start:end]
    ]

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=8192,
        return_tensors="pt"
    )

    encoded = {
        k: v.to("cuda")
        for k, v in encoded.items()
    }

    with torch.no_grad():

        output = model(**encoded)

        batch_emb = output.last_hidden_state[:, 0, :]

        batch_emb = torch.nn.functional.normalize(
            batch_emb,
            p=2,
            dim=1
        )

    embeddings[start:end] = (
        batch_emb
        .float()
        .cpu()
        .numpy()
    )

    del encoded, output, batch_emb

    # --------------------------------------------------------
    # Flush matrix FIRST, then update state
    # --------------------------------------------------------

    if end % CHECKPOINT == 0 or end == REPAIR_END:

        embeddings.flush()

        # Verify the checkpoint region before declaring progress
        check_start = end - min(CHECKPOINT, end)
        check = np.asarray(
            embeddings[check_start:end],
            dtype=np.float32
        )

        check_norms = np.linalg.norm(
            check,
            axis=1
        )

        assert np.all(
            np.isfinite(check_norms)
        )

        assert np.all(
            np.abs(check_norms - 1.0) < 1e-3
        )

        # Only NOW update state
        with open(EMB_STATE_PATH, "wb") as f:
            pickle.dump(
                {
                    "completed": end,
                    "total": N_CHUNKS,
                    "model": "BAAI/bge-m3",
                    "max_tokens": 8192,
                    "batch_size": EMB_BATCH,
                },
                f
            )

        print(
            f"Repaired: {end:,}/{REPAIR_END:,} "
            f"({100 * end / REPAIR_END:.2f}%)"
        )

    if start % 256 == 0:
        gc.collect()
        torch.cuda.empty_cache()

embeddings.flush()

print("\n" + "=" * 60)
print("REPAIR COMPLETE")
print("=" * 60)
print("Rows repaired:", REPAIR_END)
print("Remaining valid rows:", N_CHUNKS - REPAIR_END)
print("Matrix shape:", embeddings.shape)

REPAIRING MISSING EMBEDDINGS
Total chunks: 238301
Repair range: 0 -> 160000
GPU: Tesla T4
Zero rows in first 1,000: 1000
Repaired: 2,000/160,000 (1.25%)
Repaired: 4,000/160,000 (2.50%)
Repaired: 6,000/160,000 (3.75%)
Repaired: 8,000/160,000 (5.00%)
Repaired: 10,000/160,000 (6.25%)
Repaired: 12,000/160,000 (7.50%)
Repaired: 14,000/160,000 (8.75%)
Repaired: 16,000/160,000 (10.00%)
Repaired: 18,000/160,000 (11.25%)
Repaired: 20,000/160,000 (12.50%)
Repaired: 22,000/160,000 (13.75%)
Repaired: 24,000/160,000 (15.00%)
Repaired: 26,000/160,000 (16.25%)
Repaired: 28,000/160,000 (17.50%)
Repaired: 30,000/160,000 (18.75%)
Repaired: 32,000/160,000 (20.00%)
Repaired: 34,000/160,000 (21.25%)
Repaired: 36,000/160,000 (22.50%)
Repaired: 38,000/160,000 (23.75%)
Repaired: 40,000/160,000 (25.00%)
Repaired: 42,000/160,000 (26.25%)
Repaired: 44,000/160,000 (27.50%)
Repaired: 46,000/160,000 (28.75%)
Repaired: 48,000/160,000 (30.00%)
Repaired: 50,000/160,000 (31.25%)
Repaired: 52,000/160,000 (32.50%)
Repair

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 27
# FINAL FULL EMBEDDING INTEGRITY CHECK
# ============================================================

import numpy as np
from pathlib import Path

EMB_MATRIX_PATH = (
    Path("/content/drive/MyDrive/maxmin_research")
    / "cache"
    / "mldr003_embeddings"
    / "maxmin_chunk_embeddings.npy"
)

embeddings = np.load(
    EMB_MATRIX_PATH,
    mmap_mode="r"
)

print("=" * 60)
print("FINAL MAX–MIN EMBEDDING INTEGRITY")
print("=" * 60)

print("Shape:", embeddings.shape)
print("dtype:", embeddings.dtype)

assert embeddings.shape == (238301, 1024)

# ------------------------------------------------------------
# Scan entire matrix in blocks
# ------------------------------------------------------------

BLOCK = 10000

zero_rows = 0
nan_values = 0
inf_values = 0
norm_min = float("inf")
norm_max = 0.0
max_norm_error = 0.0

for start in range(0, len(embeddings), BLOCK):

    end = min(start + BLOCK, len(embeddings))

    block = np.asarray(
        embeddings[start:end],
        dtype=np.float32
    )

    nan_values += np.isnan(block).sum()
    inf_values += np.isinf(block).sum()

    norms = np.linalg.norm(block, axis=1)

    zero_rows += np.sum(norms == 0)

    norm_min = min(norm_min, float(norms.min()))
    norm_max = max(norm_max, float(norms.max()))

    max_norm_error = max(
        max_norm_error,
        float(np.max(np.abs(norms - 1.0)))
    )

print("\nFull-matrix results:")
print("  Zero rows:", zero_rows)
print("  NaN values:", nan_values)
print("  Inf values:", inf_values)
print("  Minimum L2 norm:", norm_min)
print("  Maximum L2 norm:", norm_max)
print("  Maximum |norm - 1|:", max_norm_error)

assert zero_rows == 0
assert nan_values == 0
assert inf_values == 0
assert max_norm_error < 1e-3

print("\n" + "=" * 60)
print("FINAL INTEGRITY CHECK: PASS")
print("=" * 60)

FINAL MAX–MIN EMBEDDING INTEGRITY
Shape: (238301, 1024)
dtype: float32

Full-matrix results:
  Zero rows: 0
  NaN values: 0
  Inf values: 0
  Minimum L2 norm: 0.9996225237846375
  Maximum L2 norm: 1.0003759860992432
  Maximum |norm - 1|: 0.00037747621536254883

FINAL INTEGRITY CHECK: PASS


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 28A
# Restore MLDR Query Files
# ============================================================

import os
import subprocess
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
MLDR_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://huggingface.co/datasets/Shitao/MLDR/"
    "resolve/main/mldr-v1.0-hi/"
)

FILES = {
    "dev": "dev.jsonl.gz",
    "test": "test.jsonl.gz",
}

print("=" * 60)
print("RESTORING MLDR QUERY FILES")
print("=" * 60)

for name, filename in FILES.items():

    output = MLDR_DIR / filename
    url = BASE_URL + filename + "?download=true"

    if output.exists():
        print(f"{filename}: already exists")
        continue

    print(f"Downloading {filename}...")

    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "-o",
            str(output),
            url
        ],
        check=True
    )

    print(
        f"  Saved: {output} "
        f"({output.stat().st_size / (1024**2):.2f} MB)"
    )

print("\nFiles available:")
for path in MLDR_DIR.glob("*.jsonl.gz"):
    print(" ", path.name)

print("\nSTATUS: OK")

RESTORING MLDR QUERY FILES
  Saved: /content/mldr003/dev.jsonl.gz (9.69 MB)
  Saved: /content/mldr003/test.jsonl.gz (9.84 MB)

Files available:
  dev.jsonl.gz
  test.jsonl.gz

STATUS: OK


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 28B
# Embed MLDR Hindi Queries
# ============================================================

import gzip
import json
import numpy as np
import torch
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")

MLDR_DIR = Path("/content/mldr003")

QUERY_DIR = (
    RESEARCH /
    "cache" /
    "mldr003_embeddings"
)
QUERY_DIR.mkdir(parents=True, exist_ok=True)

QUERY_EMB_PATH = QUERY_DIR / "query_embeddings.npy"
QUERY_META_PATH = QUERY_DIR / "query_metadata.json"

queries = []

for split in ["dev", "test"]:

    path = MLDR_DIR / f"{split}.jsonl.gz"

    with gzip.open(path, "rt", encoding="utf-8") as f:

        for line in f:

            row = json.loads(line)

            queries.append({
                "query_id": row["query_id"],
                "query": row["query"],
                "split": split
            })

print("=" * 60)
print("MLDR QUERY EMBEDDINGS")
print("=" * 60)

print("DEV:", sum(q["split"] == "dev" for q in queries))
print("TEST:", sum(q["split"] == "test" for q in queries))
print("Total:", len(queries))

# ------------------------------------------------------------
# Embed
# ------------------------------------------------------------

query_texts = [q["query"] for q in queries]

encoded = tokenizer(
    query_texts,
    padding=True,
    truncation=True,
    max_length=8192,
    return_tensors="pt"
)

encoded = {
    k: v.to("cuda")
    for k, v in encoded.items()
}

with torch.no_grad():

    output = model(**encoded)

    query_embeddings = output.last_hidden_state[:, 0, :]

    query_embeddings = torch.nn.functional.normalize(
        query_embeddings,
        p=2,
        dim=1
    )

query_embeddings = (
    query_embeddings
    .float()
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

np.save(
    QUERY_EMB_PATH,
    query_embeddings
)

with open(QUERY_META_PATH, "w", encoding="utf-8") as f:
    json.dump(
        queries,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# Integrity
# ------------------------------------------------------------

norms = np.linalg.norm(
    query_embeddings,
    axis=1
)

print("\nShape:", query_embeddings.shape)

print("\nNorms:")
print("  Min:", norms.min())
print("  Mean:", norms.mean())
print("  Max:", norms.max())
print(
    "  Max |norm - 1|:",
    np.max(np.abs(norms - 1.0))
)

assert query_embeddings.shape == (400, 1024)
assert np.isnan(query_embeddings).sum() == 0
assert np.isinf(query_embeddings).sum() == 0
assert np.max(np.abs(norms - 1.0)) < 1e-3

print("\nSaved:", QUERY_EMB_PATH)
print("STATUS: PASS")

MLDR QUERY EMBEDDINGS
DEV: 200
TEST: 200
Total: 400

Shape: (400, 1024)

Norms:
  Min: 0.9996978
  Mean: 0.9999962
  Max: 1.0003068
  Max |norm - 1|: 0.0003068447

Saved: /content/drive/MyDrive/maxmin_research/cache/mldr003_embeddings/query_embeddings.npy
STATUS: PASS


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 29
# MLDR Hindi TEST — Max–Min Retrieval
# ============================================================

import gzip
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
MLDR_DIR = Path("/content/mldr003")

EMB_PATH = (
    RESEARCH /
    "cache" /
    "mldr003_embeddings" /
    "maxmin_chunk_embeddings.npy"
)

QUERY_PATH = (
    RESEARCH /
    "cache" /
    "mldr003_embeddings" /
    "query_embeddings.npy"
)

QUERY_META_PATH = (
    RESEARCH /
    "cache" /
    "mldr003_embeddings" /
    "query_metadata.json"
)

print("=" * 60)
print("MLDR 003 — MAX–MIN TEST RETRIEVAL")
print("=" * 60)

# ------------------------------------------------------------
# Load embeddings + metadata
# ------------------------------------------------------------

chunk_embeddings = np.load(
    EMB_PATH,
    mmap_mode="r"
)

query_embeddings = np.load(
    QUERY_PATH
)

with open(QUERY_META_PATH, "r", encoding="utf-8") as f:
    query_metadata = json.load(f)

test_indices = [
    i for i, q in enumerate(query_metadata)
    if q["split"] == "test"
]

test_query_ids = [
    query_metadata[i]["query_id"]
    for i in test_indices
]

test_queries = query_embeddings[test_indices]

print("Chunk embeddings:", chunk_embeddings.shape)
print("TEST queries:", len(test_queries))

# ------------------------------------------------------------
# Load frozen chunk corpus
# ------------------------------------------------------------

CORPUS_PATH = (
    RESEARCH /
    "frozen" /
    "mldr003_maxmin_chunk_corpus.json"
)

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    chunk_records = json.load(f)

assert len(chunk_records) == len(chunk_embeddings)

parent_ids = np.array(
    [r["doc_id"] for r in chunk_records],
    dtype=object
)

# ------------------------------------------------------------
# Load TEST qrels
# ------------------------------------------------------------

QREL_PATH = MLDR_DIR / "test_qrels.tsv"

if not QREL_PATH.exists():

    # Restore qrels if needed
    import subprocess

    url = (
        "https://huggingface.co/datasets/Shitao/MLDR/"
        "resolve/main/qrels/qrels.mldr-v1.0-hi-test.tsv"
        "?download=true"
    )

    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "-o",
            str(QREL_PATH),
            url
        ],
        check=True
    )

qrels = pd.read_csv(
    QREL_PATH,
    sep="\t",
    header=None,
    names=[
        "query_id",
        "iteration",
        "doc_id",
        "relevance"
    ]
)

relevant = dict(
    zip(
        qrels["query_id"],
        qrels["doc_id"]
    )
)

# ------------------------------------------------------------
# Retrieval
# ------------------------------------------------------------

TOP_CHUNKS = 100

results = {}

for qi, query_id in enumerate(test_query_ids):

    q = test_queries[qi]

    # Process chunk matrix in blocks to avoid huge RAM copy
    best_scores = {}

    for start in range(0, len(chunk_embeddings), 20000):

        end = min(
            start + 20000,
            len(chunk_embeddings)
        )

        scores = np.asarray(
            chunk_embeddings[start:end] @ q,
            dtype=np.float32
        )

        # Keep top chunk candidates from this block
        k = min(TOP_CHUNKS, len(scores))

        idx = np.argpartition(
            scores,
            -k
        )[-k:]

        for local_idx in idx:

            score = float(scores[local_idx])
            chunk_idx = start + int(local_idx)
            doc_id = parent_ids[chunk_idx]

            if (
                doc_id not in best_scores
                or score > best_scores[doc_id]
            ):
                best_scores[doc_id] = score

    ranked_docs = sorted(
        best_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    ranked_docs = ranked_docs[:10]

    results[query_id] = ranked_docs

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

def reciprocal_rank(ranked, relevant_doc):
    for rank, (doc, _) in enumerate(ranked, start=1):
        if doc == relevant_doc:
            return 1.0 / rank
    return 0.0


def recall_at_k(ranked, relevant_doc, k):
    return float(
        any(
            doc == relevant_doc
            for doc, _ in ranked[:k]
        )
    )


def ndcg_at_10(ranked, relevant_doc):

    for rank, (doc, _) in enumerate(ranked[:10], start=1):

        if doc == relevant_doc:
            return 1.0 / np.log2(rank + 1)

    return 0.0


ndcg = []
mrr = []
recall1 = []
recall10 = []

for query_id in test_query_ids:

    ranked = results[query_id]
    rel = relevant[query_id]

    ndcg.append(
        ndcg_at_10(ranked, rel)
    )

    mrr.append(
        reciprocal_rank(ranked, rel)
    )

    recall1.append(
        recall_at_k(ranked, rel, 1)
    )

    recall10.append(
        recall_at_k(ranked, rel, 10)
    )

print("\n" + "=" * 60)
print("MAX–MIN TEST RESULTS")
print("=" * 60)

print(f"nDCG@10   : {np.mean(ndcg):.6f}")
print(f"MRR@10    : {np.mean(mrr):.6f}")
print(f"Recall@1  : {np.mean(recall1):.6f}")
print(f"Recall@10 : {np.mean(recall10):.6f}")

print("\nQueries evaluated:", len(test_query_ids))
print("Status: COMPLETE")

MLDR 003 — MAX–MIN TEST RETRIEVAL
Chunk embeddings: (238301, 1024)
TEST queries: 200

MAX–MIN TEST RESULTS
nDCG@10   : 0.569291
MRR@10    : 0.539020
Recall@1  : 0.480000
Recall@10 : 0.665000

Queries evaluated: 200
Status: COMPLETE


In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 30
# Prepare Native Document Corpus
# ============================================================

import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
MLDR_DIR = Path("/content/mldr003")

CORPUS_PATH = MLDR_DIR / "corpus.jsonl.gz"

print("=" * 60)
print("PREPARING NATIVE MLDR CORPUS")
print("=" * 60)

# ------------------------------------------------------------
# Load corpus
# ------------------------------------------------------------

native_docs = []

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:

    for line in f:

        row = json.loads(line)

        native_docs.append({
            "doc_id": row["doc_id"],
            "text": row["text"]
        })

print("Documents:", len(native_docs))
print(
    "Unique doc IDs:",
    len(set(d["doc_id"] for d in native_docs))
)

assert len(native_docs) == 3806
assert len(set(d["doc_id"] for d in native_docs)) == 3806

# ------------------------------------------------------------
# Token-length audit
# ------------------------------------------------------------

native_token_lengths = []

for d in native_docs:

    n = len(
        tokenizer(
            d["text"],
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    native_token_lengths.append(n)

native_token_lengths = np.asarray(
    native_token_lengths,
    dtype=np.int32
)

print("\nNative document token lengths:")

for p in [50, 90, 95, 99, 100]:
    print(
        f"  P{p}: "
        f"{np.percentile(native_token_lengths, p):.0f}"
    )

print("\nDocuments exceeding BGE-M3 limit:")

for threshold in [4096, 8192, 16384, 32768]:
    count = np.sum(native_token_lengths > threshold)
    print(
        f"  >{threshold}: "
        f"{count} "
        f"({100 * count / len(native_docs):.2f}%)"
    )

print("\nMaximum:", native_token_lengths.max())

# ------------------------------------------------------------
# Save frozen native corpus
# ------------------------------------------------------------

NATIVE_FROZEN = (
    RESEARCH /
    "frozen" /
    "mldr003_native_corpus.json"
)

with open(NATIVE_FROZEN, "w", encoding="utf-8") as f:
    json.dump(
        native_docs,
        f,
        ensure_ascii=False,
        separators=(",", ":")
    )

print("\nSaved:", NATIVE_FROZEN)
print("STATUS: READY")

PREPARING NATIVE MLDR CORPUS


KeyError: 'doc_id'

In [ ]:
# ============================================================
# EXPERIMENT 003 — CELL 30A
# Restore MLDR Hindi Corpus
# ============================================================

import subprocess
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
MLDR_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = MLDR_DIR / "corpus.jsonl.gz"

URL = (
    "https://huggingface.co/datasets/Shitao/MLDR/"
    "resolve/main/mldr-v1.0-hi/corpus.jsonl.gz"
    "?download=true"
)

print("=" * 60)
print("RESTORING MLDR CORPUS")
print("=" * 60)

if CORPUS_PATH.exists():

    print("Corpus already exists.")
    print(
        f"Size: {CORPUS_PATH.stat().st_size / (1024**2):.2f} MB"
    )

else:

    print("Downloading corpus...")

    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "-o",
            str(CORPUS_PATH),
            URL
        ],
        check=True
    )

    print(
        f"Downloaded: "
        f"{CORPUS_PATH.stat().st_size / (1024**2):.2f} MB"
    )

print("\nSTATUS: OK")

RESTORING MLDR CORPUS
Downloaded: 35.16 MB

STATUS: OK


In [ ]:
import gzip
import json

print("Inspecting first MLDR corpus record...\n")

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    row = json.loads(f.readline())

print("Keys:", list(row.keys()))
print("\nFirst record:")
print(row)

Inspecting first MLDR corpus record...

Keys: ['docid', 'text']

First record:
{'docid': 'doc-hi-428', 'text': "उज्जैन\n[[चित्र:Mahakal temple.jpg|250px|right|thumb|उज्जैन का प्रसिद्ध महाकालेश्वर मंदिर]]\nउज्जैन (उज्जयिनी) भारत के मध्य प्रदेश राज्य का एक प्रमुख शहर है जो क्षिप्रा नदी या शिप्रा नदी के किनारे पर बसा है। यह एक अत्यन्त प्राचीन शहर है। यह महान सम्राट विक्रमादित्य के राज्य की राजधानी थी ।  उज्जैन को कालिदास की नगरी के नाम से भी जाना जाता है। यहाँ हर 12 वर्ष पर सिंहस्थ महाकुंभ मेला लगता है। भगवान शिव के 12 ज्योतिर्लिंगों में एक महाकाल इस नगरी में स्थित है। उज्जैन मध्य प्रदेश के सबसे बड़े शहर इन्दौर से 45 कि॰मी॰ पर है। उज्जैन के प्राचीन नाम अवन्तिका, उज्जयनी, कनकश्रन्गा आदि है। उज्जैन मंदिरों की नगरी है। यहाँ कई तीर्थ स्थल है। इसकी जनसंख्या 515215  लाख सन ,2011 की जनगणना के हिसाब से  है। यह मध्य प्रदेश का पाँचवा सबसे बड़ा शहर है। नगर निगम सीमा का क्षेत्रफल 152 वर्ग किलोमीटर है।\n\nदर्शनीय स्थल\nचिन्तामन\tगणेश\nस्थिरमन\tगणेश\nगढ़कालिका\tमाताजी\nहरसिद्धि\tमाताजी\nकालभैरव\tदेव\nम

In [ ]:
import gzip
import json
from pathlib import Path

print("=" * 60)
print("PREPARING NATIVE MLDR CORPUS")
print("=" * 60)

native_docs = []

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        native_docs.append({
            "doc_id": row["docid"],   # MLDR uses 'docid'
            "text": row["text"]
        })

print(f"\nDocuments loaded: {len(native_docs):,}")
print(f"Unique doc IDs: {len(set(d['doc_id'] for d in native_docs)):,}")

print("\nFirst document:")
print("ID:", native_docs[0]["doc_id"])
print("Text length:", len(native_docs[0]["text"]))

assert len(native_docs) == 3806
assert len(set(d["doc_id"] for d in native_docs)) == 3806

print("\nSTATUS: OK")

PREPARING NATIVE MLDR CORPUS

Documents loaded: 3,806
Unique doc IDs: 3,806

First document:
ID: doc-hi-428
Text length: 16649

STATUS: OK


In [ ]:
print("=" * 60)
print("NATIVE MLDR BGE-M3 TOKEN AUDIT")
print("=" * 60)

native_token_lengths = []

for d in native_docs:
    n = len(tokenizer(d["text"], add_special_tokens=True)["input_ids"])
    native_token_lengths.append(n)

native_token_lengths = np.array(native_token_lengths)

print(f"\nDocuments: {len(native_token_lengths):,}")
print(f"Mean:      {native_token_lengths.mean():,.2f}")
print(f"Median:    {np.median(native_token_lengths):,.2f}")
print(f"P90:       {np.percentile(native_token_lengths, 90):,.2f}")
print(f"P95:       {np.percentile(native_token_lengths, 95):,.2f}")
print(f"P99:       {np.percentile(native_token_lengths, 99):,.2f}")
print(f"Max:       {native_token_lengths.max():,}")

for limit in [4096, 8192, 12288, 16384, 32768]:
    print(
        f"> {limit:5,}: "
        f"{np.sum(native_token_lengths > limit):4,} "
        f"({100*np.mean(native_token_lengths > limit):.2f}%)"
    )

NATIVE MLDR BGE-M3 TOKEN AUDIT


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8943 > 8192). Running this sequence through the model will result in indexing errors



Documents: 3,806
Mean:      5,555.04
Median:    3,565.50
P90:       12,120.50
P95:       15,327.00
P99:       22,126.55
Max:       92,405
> 4,096: 1,801 (47.32%)
> 8,192:  955 (25.09%)
> 12,288:  368 (9.67%)
> 16,384:  148 (3.89%)
> 32,768:    6 (0.16%)


In [ ]:
print("=" * 60)
print("PREPARING NATIVE BGE-M3 TEXTS")
print("=" * 60)

NATIVE_MAX_TOKENS = 8192

native_texts = []
native_truncated = []

for d, n_tokens in zip(native_docs, native_token_lengths):
    if n_tokens > NATIVE_MAX_TOKENS:
        ids = tokenizer(
            d["text"],
            add_special_tokens=True,
            truncation=True,
            max_length=NATIVE_MAX_TOKENS
        )["input_ids"]

        text = tokenizer.decode(
            ids,
            skip_special_tokens=True
        )

        native_truncated.append(True)
    else:
        text = d["text"]
        native_truncated.append(False)

    native_texts.append(text)

native_truncated = np.array(native_truncated)

print(f"\nDocuments: {len(native_texts):,}")
print(f"Truncated: {native_truncated.sum():,}")
print(f"Untruncated: {(~native_truncated).sum():,}")

assert len(native_texts) == 3806
assert native_truncated.sum() == 955

print("\nSTATUS: OK")

PREPARING NATIVE BGE-M3 TEXTS

Documents: 3,806
Truncated: 955
Untruncated: 2,851

STATUS: OK


In [ ]:
print("=" * 60)
print("EMBEDDING NATIVE MLDR CORPUS")
print("=" * 60)

NATIVE_EMB_PATH = RESEARCH / "cache" / "mldr003_embeddings" / "native_embeddings.npy"
NATIVE_EMB_PATH.parent.mkdir(parents=True, exist_ok=True)

NATIVE_BATCH_SIZE = 8
N = len(native_texts)
EMB_DIM = 1024

native_embeddings = np.zeros((N, EMB_DIM), dtype=np.float32)

for start in range(0, N, NATIVE_BATCH_SIZE):
    end = min(start + NATIVE_BATCH_SIZE, N)

    batch = native_texts[start:end]

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=8192,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    emb = outputs.last_hidden_state[:, 0, :]
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)

    native_embeddings[start:end] = emb.cpu().numpy()

    if end % 200 == 0 or end == N:
        print(f"Embedded: {end:,}/{N:,}")

np.save(NATIVE_EMB_PATH, native_embeddings)

print("\nSaved:", NATIVE_EMB_PATH)
print("Shape:", native_embeddings.shape)
print("STATUS: EMBEDDING COMPLETE")

EMBEDDING NATIVE MLDR CORPUS
Embedded: 200/3,806


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive")
RESEARCH = BASE / "maxmin_research"
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
FROZEN = RESEARCH / "frozen"
RESULTS = RESEARCH / "results"

print("=" * 55)
print("EXPERIMENT 003 — RESUME CHECK")
print("=" * 55)

# GPU status
try:
    import torch
    print(f"\nGPU available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print("PyTorch check:", e)

# Required cached artifacts
files = [
    CACHE / "maxmin_chunk_embeddings.npy",
    CACHE / "maxmin_embedding_state.pkl",
    CACHE / "query_embeddings.npy",
    CACHE / "query_metadata.json",
    FROZEN / "mldr003_maxmin_chunk_corpus.json",
]

print("\nCached artifacts:")
for path in files:
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"[OK] {path.name}: {size_mb:.2f} MB")
    else:
        print(f"[MISSING] {path.name}")

# Native embedding checkpoint
native_path = CACHE / "native_embeddings.npy"
if native_path.exists():
    print(f"\n[EXISTS] Native embeddings: "
          f"{native_path.stat().st_size / (1024**2):.2f} MB")
else:
    print("\nNative embeddings: not saved yet")

print("\nSTATUS: INSPECTION ONLY — no embeddings computed")

EXPERIMENT 003 — RESUME CHECK

GPU available: True
GPU: Tesla T4

Cached artifacts:
[MISSING] maxmin_chunk_embeddings.npy
[MISSING] maxmin_embedding_state.pkl
[MISSING] query_embeddings.npy
[MISSING] query_metadata.json
[MISSING] mldr003_maxmin_chunk_corpus.json

Native embeddings: not saved yet

STATUS: INSPECTION ONLY — no embeddings computed


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive")
RESEARCH = BASE / "maxmin_research"

print("Drive mounted:", BASE.exists())
print("Research folder exists:", RESEARCH.exists())

print("\nMyDrive contents:")
if BASE.exists():
    for p in sorted(BASE.iterdir()):
        print(f"  {'[DIR] ' if p.is_dir() else '[FILE] '}{p.name}")

if RESEARCH.exists():
    print("\nResearch folder contents:")
    for p in sorted(RESEARCH.iterdir()):
        print(f"  {'[DIR] ' if p.is_dir() else '[FILE] '}{p.name}")

Drive mounted: False
Research folder exists: False

MyDrive contents:


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive")
RESEARCH = BASE / "maxmin_research"

print("MyDrive exists:", BASE.exists())
print("Research folder exists:", RESEARCH.exists())

if RESEARCH.exists():
    for folder in sorted(RESEARCH.iterdir()):
        print(f"{'[DIR] ' if folder.is_dir() else '[FILE] '}{folder.name}")

MyDrive exists: True
Research folder exists: True
[DIR] cache
[DIR] frozen
[DIR] results


In [ ]:
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")

for folder_name in ["cache", "frozen", "results"]:
    folder = RESEARCH / folder_name
    print(f"\n{'=' * 55}\n{folder_name.upper()}\n{'=' * 55}")

    for path in sorted(folder.rglob("*")):
        if path.is_file():
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"{size_mb:9.2f} MB | {path.relative_to(RESEARCH)}")



CACHE
    70.98 MB | cache/bge_m3_fixed2_embeddings/bge_m3_fixed2_embeddings.npy
     0.53 MB | cache/bge_m3_fixed2_embeddings/fixed2_chunk_ids.json
    44.22 MB | cache/bge_m3_fixed2_embeddings/new_unique_text_embeddings.npy
     0.73 MB | cache/bge_m3_fixed2_embeddings/new_unique_text_hashes.json
    65.48 MB | cache/bge_m3_maxmin_embeddings/bge_m3_maxmin_embeddings.npy
     0.56 MB | cache/bge_m3_maxmin_embeddings/maxmin_chunk_ids.json
     9.40 MB | cache/bge_m3_maxmin_embeddings/new_unique_text_embeddings.npy
     0.16 MB | cache/bge_m3_maxmin_embeddings/new_unique_text_hashes.json
    35.05 MB | cache/bge_m3_native_embeddings/bge_m3_native_embeddings.npy
     0.19 MB | cache/bge_m3_native_embeddings/parent_ids.json
     1.79 MB | cache/bge_m3_query_embeddings/bge_m3_query_embeddings.npy
     0.01 MB | cache/bge_m3_query_embeddings/query_keys.json
    65.48 MB | cache/bge_m3_random_boundary_embeddings/seed42_bge_random_embeddings.npy
    11.38 MB | cache/bge_m3_random_boundary_em

In [ ]:
import numpy as np
import json
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"

CHUNK_EMB_PATH = CACHE / "maxmin_chunk_embeddings.npy"
QUERY_EMB_PATH = CACHE / "query_embeddings.npy"
CHUNK_CORPUS_PATH = RESEARCH / "frozen" / "mldr003_maxmin_chunk_corpus.json"

print("=" * 55)
print("EXPERIMENT 003 — EMBEDDING INTEGRITY CHECK")
print("=" * 55)

chunk_embeddings = np.load(CHUNK_EMB_PATH, mmap_mode="r")
query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode="r")

print("\nMax-Min embeddings:", chunk_embeddings.shape)
print("Query embeddings:", query_embeddings.shape)

print("\nChecking chunk embedding integrity...")
zero_rows = 0
nan_count = 0
inf_count = 0
norm_min = float("inf")
norm_max = 0.0

for start in range(0, len(chunk_embeddings), 10000):
    batch = np.asarray(chunk_embeddings[start:start+10000])
    norms = np.linalg.norm(batch, axis=1)

    zero_rows += int(np.sum(norms == 0))
    nan_count += int(np.isnan(batch).sum())
    inf_count += int(np.isinf(batch).sum())

    norm_min = min(norm_min, float(norms.min()))
    norm_max = max(norm_max, float(norms.max()))

print("Zero rows:", zero_rows)
print("NaN values:", nan_count)
print("Inf values:", inf_count)
print(f"Norm range: {norm_min:.6f} – {norm_max:.6f}")

print("\nChunk corpus exists:", CHUNK_CORPUS_PATH.exists())
print("STATUS:", "PASS" if (
    chunk_embeddings.shape == (238301, 1024)
    and query_embeddings.shape == (400, 1024)
    and zero_rows == 0
    and nan_count == 0
    and inf_count == 0
    and abs(norm_min - 1) < 0.001
    and abs(norm_max - 1) < 0.001
    and CHUNK_CORPUS_PATH.exists()
) else "CHECK REQUIRED")

EXPERIMENT 003 — EMBEDDING INTEGRITY CHECK

Max-Min embeddings: (238301, 1024)
Query embeddings: (400, 1024)

Checking chunk embedding integrity...
Zero rows: 0
NaN values: 0
Inf values: 0
Norm range: 0.999623 – 1.000376

Chunk corpus exists: True
STATUS: PASS


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

print("\nModel loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Native texts in memory:", "native_texts" in globals())
print("Native token lengths in memory:", "native_token_lengths" in globals())

GPU available: True
GPU: Tesla T4
GPU memory allocated: 0.00 GB
GPU memory reserved:  0.00 GB

Model loaded: False
Tokenizer loaded: False
Native texts in memory: False
Native token lengths in memory: False


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "BAAI/bge-m3"

print("Loading BGE-M3...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
).to("cuda")

model.eval()

print("\nModel:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Embedding dimension:", model.config.hidden_size)
print("Max model tokens:", tokenizer.model_max_length)
print("GPU:", torch.cuda.get_device_name(0))
print("\nSTATUS: MODEL READY")

Loading BGE-M3...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            


Model: BAAI/bge-m3
Device: cuda:0
Embedding dimension: 1024
Max model tokens: 8192
GPU: Tesla T4

STATUS: MODEL READY


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"

MLDR_DIR = Path("/content/mldr003")
CORPUS_PATH = MLDR_DIR / "corpus.jsonl.gz"

print("=" * 55)
print("RESTORING NATIVE CORPUS")
print("=" * 55)

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Corpus not found at {CORPUS_PATH}. "
        "We'll download it again if necessary."
    )

native_docs = []

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        native_docs.append({
            "doc_id": row["docid"],
            "text": row["text"]
        })

assert len(native_docs) == 3806
assert len({d["doc_id"] for d in native_docs}) == 3806

print("Documents:", len(native_docs))
print("First document:", native_docs[0]["doc_id"])
print("STATUS: CORPUS RESTORED")

RESTORING NATIVE CORPUS


FileNotFoundError: Corpus not found at /content/mldr003/corpus.jsonl.gz. We'll download it again if necessary.

In [ ]:
import subprocess
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
MLDR_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = MLDR_DIR / "corpus.jsonl.gz"

URL = (
    "https://huggingface.co/datasets/Shitao/MLDR/"
    "resolve/main/mldr-v1.0-hi/corpus.jsonl.gz?download=true"
)

print("Downloading MLDR Hindi corpus...")

subprocess.run(
    ["curl", "-L", "--fail", "--silent", "--show-error",
     "-o", str(CORPUS_PATH), URL],
    check=True
)

size_mb = CORPUS_PATH.stat().st_size / (1024 ** 2)

print(f"\nDownloaded: {size_mb:.2f} MB")
print("Saved temporarily to:", CORPUS_PATH)
print("STATUS:", "OK" if size_mb > 30 else "CHECK DOWNLOAD")


Downloaded: 35.16 MB
Saved temporarily to: /content/mldr003/corpus.jsonl.gz
STATUS: OK


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
CORPUS_PATH = Path("/content/mldr003/corpus.jsonl.gz")

print("Loading MLDR Hindi corpus...")

native_docs = []

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        native_docs.append({
            "doc_id": row["docid"],
            "text": row["text"]
        })

assert len(native_docs) == 3806
assert len({d["doc_id"] for d in native_docs}) == 3806

print("Documents:", len(native_docs))

print("\nTokenizing Native documents...")
native_texts = []
native_token_lengths = []

for d in native_docs:
    ids = tokenizer(
        d["text"],
        add_special_tokens=True,
        truncation=True,
        max_length=8192
    )["input_ids"]

    native_token_lengths.append(len(tokenizer(
        d["text"], add_special_tokens=True
    )["input_ids"]))

    # Match the previous protocol: truncate to 8192 tokens.
    native_texts.append(tokenizer.decode(ids, skip_special_tokens=True))

native_token_lengths = np.array(native_token_lengths)
native_truncated = native_token_lengths > 8192

print("Native texts prepared:", len(native_texts))
print("Documents over 8192 tokens:", int(native_truncated.sum()))
print("Prepared text maximum tokens:", max(
    len(tokenizer(t, add_special_tokens=True)["input_ids"])
    for t in native_texts
))
print("\nSTATUS: READY")

Loading MLDR Hindi corpus...
Documents: 3806

Tokenizing Native documents...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8943 > 8192). Running this sequence through the model will result in indexing errors


Native texts prepared: 3806
Documents over 8192 tokens: 955
Prepared text maximum tokens: 8220

STATUS: READY


In [ ]:
import json
import numpy as np
import torch
from pathlib import Path

NATIVE_EMB_PATH = CACHE / "native_embeddings.npy"
NATIVE_STATE_PATH = CACHE / "native_embedding_state.json"

N = len(native_texts)
DIM = 1024
BATCH_SIZE = 2
CHECKPOINT_EVERY = 100
MAX_TOKENS = 8192

# Resume only if both artifacts agree on the matrix shape and progress.
if NATIVE_EMB_PATH.exists() and NATIVE_STATE_PATH.exists():
    with open(NATIVE_STATE_PATH, "r") as f:
        state = json.load(f)

    if state.get("n_docs") != N or state.get("dim") != DIM:
        raise RuntimeError("Native checkpoint metadata mismatch.")

    native_embeddings = np.load(NATIVE_EMB_PATH, mmap_mode="r+")
    if native_embeddings.shape != (N, DIM):
        raise RuntimeError("Native embedding matrix shape mismatch.")

    start_at = int(state["completed"])
    print(f"Resuming from document {start_at:,}/{N:,}")

else:
    native_embeddings = np.lib.format.open_memmap(
        NATIVE_EMB_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(N, DIM)
    )
    start_at = 0

    with open(NATIVE_STATE_PATH, "w") as f:
        json.dump(
            {"n_docs": N, "dim": DIM, "completed": 0},
            f
        )

    print("Starting fresh Native embedding run.")

def save_native_progress(completed):
    native_embeddings.flush()
    tmp_path = NATIVE_STATE_PATH.with_suffix(".tmp")
    with open(tmp_path, "w") as f:
        json.dump(
            {"n_docs": N, "dim": DIM, "completed": completed},
            f
        )
    tmp_path.replace(NATIVE_STATE_PATH)

print(f"Embedding {N - start_at:,} remaining documents...")
print(f"Batch size: {BATCH_SIZE}; max input tokens: {MAX_TOKENS}")

for start in range(start_at, N, BATCH_SIZE):
    end = min(start + BATCH_SIZE, N)
    batch = native_texts[start:end]

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=MAX_TOKENS,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state[:, 0, :]
        emb = torch.nn.functional.normalize(emb.float(), p=2, dim=1)

    native_embeddings[start:end] = emb.cpu().numpy()

    completed = end
    if completed % CHECKPOINT_EVERY < BATCH_SIZE or completed == N:
        save_native_progress(completed)
        print(f"Checkpoint: {completed:,}/{N:,}")

save_native_progress(N)

print("\nNative embedding complete.")
print("Shape:", native_embeddings.shape)
print("Saved:", NATIVE_EMB_PATH)
print("STATUS: COMPLETE")

Starting fresh Native embedding run.
Embedding 3,806 remaining documents...
Batch size: 2; max input tokens: 8192
Checkpoint: 100/3,806
Checkpoint: 200/3,806
Checkpoint: 300/3,806
Checkpoint: 400/3,806
Checkpoint: 500/3,806
Checkpoint: 600/3,806
Checkpoint: 700/3,806
Checkpoint: 800/3,806
Checkpoint: 900/3,806
Checkpoint: 1,000/3,806
Checkpoint: 1,100/3,806
Checkpoint: 1,200/3,806
Checkpoint: 1,300/3,806
Checkpoint: 1,400/3,806
Checkpoint: 1,500/3,806
Checkpoint: 1,600/3,806
Checkpoint: 1,700/3,806
Checkpoint: 1,800/3,806
Checkpoint: 1,900/3,806
Checkpoint: 2,000/3,806
Checkpoint: 2,100/3,806
Checkpoint: 2,200/3,806
Checkpoint: 2,300/3,806
Checkpoint: 2,400/3,806
Checkpoint: 2,500/3,806
Checkpoint: 2,600/3,806
Checkpoint: 2,700/3,806
Checkpoint: 2,800/3,806
Checkpoint: 2,900/3,806
Checkpoint: 3,000/3,806
Checkpoint: 3,100/3,806
Checkpoint: 3,200/3,806
Checkpoint: 3,300/3,806
Checkpoint: 3,400/3,806
Checkpoint: 3,500/3,806
Checkpoint: 3,600/3,806
Checkpoint: 3,700/3,806
Checkpoint: 3,80

In [ ]:
import numpy as np
from pathlib import Path

NATIVE_EMB_PATH = (
    Path("/content/drive/MyDrive/maxmin_research")
    / "cache/mldr003_embeddings/native_embeddings.npy"
)

native_embeddings = np.load(NATIVE_EMB_PATH, mmap_mode="r")

print("=" * 50)
print("NATIVE EMBEDDING VALIDATION")
print("=" * 50)

print("Shape:", native_embeddings.shape)

assert native_embeddings.shape == (3806, 1024)

zero_rows = 0
nan_count = 0
inf_count = 0
norm_min = float("inf")
norm_max = 0.0

for start in range(0, len(native_embeddings), 500):
    batch = np.asarray(native_embeddings[start:start + 500])
    norms = np.linalg.norm(batch, axis=1)

    zero_rows += int(np.sum(norms == 0))
    nan_count += int(np.isnan(batch).sum())
    inf_count += int(np.isinf(batch).sum())

    norm_min = min(norm_min, float(norms.min()))
    norm_max = max(norm_max, float(norms.max()))

print("Zero rows:", zero_rows)
print("NaN values:", nan_count)
print("Inf values:", inf_count)
print(f"Norm range: {norm_min:.6f} – {norm_max:.6f}")

assert zero_rows == 0
assert nan_count == 0
assert inf_count == 0
assert abs(norm_min - 1.0) < 0.001
assert abs(norm_max - 1.0) < 0.001

print("\nSTATUS: PASS")

NATIVE EMBEDDING VALIDATION
Shape: (3806, 1024)
Zero rows: 0
NaN values: 0
Inf values: 0
Norm range: 1.000000 – 1.000000

STATUS: PASS


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
MLDR_DIR = Path("/content/mldr003")

# Restore query embeddings and metadata
query_embeddings = np.load(CACHE / "query_embeddings.npy")
with open(CACHE / "query_metadata.json", "r", encoding="utf-8") as f:
    query_metadata = json.load(f)

# Restore native embeddings
native_embeddings = np.load(CACHE / "native_embeddings.npy", mmap_mode="r")

# Restore corpus in its original order
CORPUS_PATH = MLDR_DIR / "corpus.jsonl.gz"
if not CORPUS_PATH.exists():
    raise FileNotFoundError("MLDR corpus missing from temporary Colab storage.")

native_docs = []
with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        native_docs.append({
            "doc_id": row["docid"],
            "text": row["text"]
        })

assert len(native_docs) == 3806
assert native_embeddings.shape == (3806, 1024)

# Restore TEST queries and qrels
TEST_PATH = MLDR_DIR / "test.jsonl.gz"
QRELS_PATH = MLDR_DIR / "qrels-test.tsv"

if not TEST_PATH.exists() or not QRELS_PATH.exists():
    raise FileNotFoundError(
        "TEST queries or qrels missing from /content/mldr003."
    )

test_queries = {}
with gzip.open(TEST_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        test_queries[str(row["query_id"])] = row["query"]

qrels = {}
with open(QRELS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4:
            qid, _, doc_id, relevance = parts[:4]
            if int(relevance) > 0:
                qrels.setdefault(str(qid), set()).add(doc_id)

# Metadata can vary slightly in serialization; inspect rather than guess.
print("Query metadata type:", type(query_metadata).__name__)
print("Query metadata preview:")
print(query_metadata[:2] if isinstance(query_metadata, list)
      else list(query_metadata.items())[:2])

print("\nTEST queries:", len(test_queries))
print("TEST qrels:", len(qrels))
print("Native docs:", len(native_docs))
print("Native embeddings:", native_embeddings.shape)

FileNotFoundError: TEST queries or qrels missing from /content/mldr003.

In [ ]:
import subprocess
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
MLDR_DIR.mkdir(parents=True, exist_ok=True)

downloads = {
    "test.jsonl.gz": (
        "https://huggingface.co/datasets/Shitao/MLDR/"
        "resolve/main/mldr-v1.0-hi/test.jsonl.gz?download=true"
    ),
    "qrels-test.tsv": (
        "https://huggingface.co/datasets/Shitao/MLDR/"
        "resolve/main/qrels/qrels.mldr-v1.0-hi-test.tsv?download=true"
    ),
}

for filename, url in downloads.items():
    path = MLDR_DIR / filename

    print(f"Downloading {filename}...")
    subprocess.run(
        [
            "curl", "-L", "--fail", "--silent", "--show-error",
            "-o", str(path), url
        ],
        check=True
    )

    print(f"  Saved: {path}")
    print(f"  Size: {path.stat().st_size / 1024:.1f} KB")

print("\nSTATUS: DOWNLOADS COMPLETE")

  Saved: /content/mldr003/test.jsonl.gz
  Size: 10075.9 KB
  Saved: /content/mldr003/qrels-test.tsv
  Size: 5.0 KB

STATUS: DOWNLOADS COMPLETE


In [ ]:
import gzip
import json
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
CACHE = Path("/content/drive/MyDrive/maxmin_research/cache/mldr003_embeddings")

# Load TEST queries
test_queries = {}

with gzip.open(MLDR_DIR / "test.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        test_queries[str(row["query_id"])] = row["query"]

# Load qrels
qrels = {}

with open(MLDR_DIR / "qrels-test.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4:
            qid, _, doc_id, relevance = parts[:4]
            if int(relevance) > 0:
                qrels.setdefault(str(qid), set()).add(doc_id)

# Load saved query metadata
with open(CACHE / "query_metadata.json", "r", encoding="utf-8") as f:
    query_metadata = json.load(f)

print("=" * 55)
print("MLDR TEST ALIGNMENT AUDIT")
print("=" * 55)

print("Unique TEST queries:", len(test_queries))
print("Unique qrel query IDs:", len(qrels))
print("Queries with relevant documents:", len(qrels))
print("Query metadata type:", type(query_metadata).__name__)

if isinstance(query_metadata, list):
    print("Metadata entries:", len(query_metadata))
    print("First entries:", query_metadata[:3])
elif isinstance(query_metadata, dict):
    print("Metadata entries:", len(query_metadata))
    print("First entries:", list(query_metadata.items())[:3])

print("\nQuery IDs missing qrels:", len(set(test_queries) - set(qrels)))
print("Qrel IDs missing queries:", len(set(qrels) - set(test_queries)))
print("Queries with exactly one relevant document:",
      sum(len(v) == 1 for v in qrels.values()))

assert len(test_queries) == 200
assert len(qrels) == 200
assert set(test_queries) == set(qrels)

print("\nSTATUS: QUERY/QREL IDS ALIGNED")

MLDR TEST ALIGNMENT AUDIT
Unique TEST queries: 200
Unique qrel query IDs: 200
Queries with relevant documents: 200
Query metadata type: list
Metadata entries: 400
First entries: [{'query_id': 'q-hi-9', 'query': 'श्रीगंगनगर सुगर मिल्स लिमिटेड में एक योजना १९६८ में आरम्भ की गई थी।', 'split': 'dev'}, {'query_id': 'q-hi-19', 'query': 'बिजली खाँ पठान द्वारा राजा बीर सिंह बघेल के देश में किस प्रकार का बदलाव लाया गया था?', 'split': 'dev'}, {'query_id': 'q-hi-21', 'query': 'कौन सी तकनीकी योजनाएं हैं जो इन सांस्कृतिक सुविधाओं को संचालित करती हैं?', 'split': 'dev'}]

Query IDs missing qrels: 0
Qrel IDs missing queries: 0
Queries with exactly one relevant document: 200

STATUS: QUERY/QREL IDS ALIGNED


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
MLDR_DIR = Path("/content/mldr003")
RESULTS = RESEARCH / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# Load embeddings
native_embeddings = np.load(CACHE / "native_embeddings.npy", mmap_mode="r")
query_embeddings = np.load(CACHE / "query_embeddings.npy")

with open(CACHE / "query_metadata.json", "r", encoding="utf-8") as f:
    query_metadata = json.load(f)

# Load Native document IDs in embedding order
with gzip.open(MLDR_DIR / "corpus.jsonl.gz", "rt", encoding="utf-8") as f:
    native_doc_ids = [json.loads(line)["docid"] for line in f]

# Load TEST queries
test_queries = {}
with gzip.open(MLDR_DIR / "test.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        test_queries[str(row["query_id"])] = row["query"]

# Load relevance labels
qrels = {}
with open(MLDR_DIR / "qrels-test.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4 and int(parts[3]) > 0:
            qrels.setdefault(str(parts[0]), set()).add(parts[2])

# Explicit query-ID -> embedding-row mapping
query_row = {
    str(item["query_id"]): i
    for i, item in enumerate(query_metadata)
}

test_ids = sorted(test_queries)

assert native_embeddings.shape == (3806, 1024)
assert len(native_doc_ids) == 3806
assert len(set(native_doc_ids)) == 3806
assert query_embeddings.shape == (400, 1024)
assert all(qid in query_row for qid in test_ids)
assert set(test_queries) == set(qrels)

# Retrieve exact top-10 documents for each TEST query.
# Full-corpus matrix multiplication; no approximate retrieval.
doc_matrix = np.asarray(native_embeddings, dtype=np.float32)
doc_id_to_row = {doc_id: i for i, doc_id in enumerate(native_doc_ids)}

per_query = []

for qid in test_ids:
    qi = query_row[qid]
    qvec = query_embeddings[qi].astype(np.float32)

    scores = doc_matrix @ qvec
    top_idx = np.argsort(-scores)[:10]
    ranked_ids = [native_doc_ids[i] for i in top_idx]

    relevant = qrels[qid]
    hits = np.array([1 if d in relevant else 0 for d in ranked_ids])

    # Binary relevance; MLDR TEST has one relevant document per query.
    discounts = 1.0 / np.log2(np.arange(2, 12))
    dcg = float(np.sum(hits * discounts))
    idcg = 1.0  # One relevant document; ideal rank is 1.
    ndcg = dcg / idcg

    hit_positions = np.flatnonzero(hits)
    mrr = 1.0 / (int(hit_positions[0]) + 1) if len(hit_positions) else 0.0

    # Save the rank of the relevant document, if retrieved.
    relevant_rank = next(
        (i + 1 for i, d in enumerate(ranked_ids) if d in relevant),
        None
    )

    per_query.append({
        "query_id": qid,
        "ndcg@10": ndcg,
        "mrr@10": mrr,
        "recall@1": float(hits[0]),
        "recall@10": float(np.any(hits)),
        "relevant_rank": relevant_rank,
        "top10_doc_ids": ranked_ids
    })

metrics = {
    "n_queries": len(per_query),
    "ndcg@10": float(np.mean([r["ndcg@10"] for r in per_query])),
    "mrr@10": float(np.mean([r["mrr@10"] for r in per_query])),
    "recall@1": float(np.mean([r["recall@1"] for r in per_query])),
    "recall@10": float(np.mean([r["recall@10"] for r in per_query]))
}

out_path = RESULTS / "mldr003_native_test_per_query.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {"metrics": metrics, "per_query": per_query},
        f, ensure_ascii=False, indent=2
    )

print("=" * 55)
print("EXPERIMENT 003 — NATIVE TEST RESULTS")
print("=" * 55)

for metric, value in metrics.items():
    print(f"{metric}: {value:.6f}" if isinstance(value, float)
          else f"{metric}: {value}")

print("\nSaved:", out_path)
print("STATUS: COMPLETE")


EXPERIMENT 003 — NATIVE TEST RESULTS
n_queries: 200
ndcg@10: 0.409589
mrr@10: 0.372262
recall@1: 0.310000
recall@10: 0.530000

Saved: /content/drive/MyDrive/maxmin_research/results/mldr003_native_test_per_query.json
STATUS: COMPLETE


In [ ]:
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
FROZEN = RESEARCH / "frozen"
MLDR_DIR = Path("/content/mldr003")

paths = {
    "Max-Min chunk corpus": FROZEN / "mldr003_maxmin_chunk_corpus.json",
    "Max-Min embeddings": CACHE / "maxmin_chunk_embeddings.npy",
    "Query embeddings": CACHE / "query_embeddings.npy",
    "Query metadata": CACHE / "query_metadata.json",
    "Native per-query results": RESEARCH / "results" / "mldr003_native_test_per_query.json",
    "TEST queries": MLDR_DIR / "test.jsonl.gz",
    "TEST qrels": MLDR_DIR / "qrels-test.tsv",
}

print("=" * 55)
print("EXPERIMENT 003 — MAX-MIN RETRIEVAL RECHECK")
print("=" * 55)

for name, path in paths.items():
    print(f"{'[OK]' if path.exists() else '[MISSING]'} {name}: {path}")

assert all(path.exists() for path in paths.values()), (
    "One or more inputs are missing. Restore them before retrieval."
)

chunk_embeddings = np.load(paths["Max-Min embeddings"], mmap_mode="r")
query_embeddings = np.load(paths["Query embeddings"], mmap_mode="r")

with open(paths["Max-Min chunk corpus"], "r", encoding="utf-8") as f:
    chunk_corpus = json.load(f)

print("\nChunk embedding shape:", chunk_embeddings.shape)
print("Query embedding shape:", query_embeddings.shape)
print("Chunk corpus type:", type(chunk_corpus).__name__)
print("Chunk corpus entries:", len(chunk_corpus))

assert chunk_embeddings.shape == (238301, 1024)
assert query_embeddings.shape == (400, 1024)
assert len(chunk_corpus) == 238301

print("\nSTATUS: INPUTS VERIFIED")

EXPERIMENT 003 — MAX-MIN RETRIEVAL RECHECK
[OK] Max-Min chunk corpus: /content/drive/MyDrive/maxmin_research/frozen/mldr003_maxmin_chunk_corpus.json
[OK] Max-Min embeddings: /content/drive/MyDrive/maxmin_research/cache/mldr003_embeddings/maxmin_chunk_embeddings.npy
[OK] Query embeddings: /content/drive/MyDrive/maxmin_research/cache/mldr003_embeddings/query_embeddings.npy
[OK] Query metadata: /content/drive/MyDrive/maxmin_research/cache/mldr003_embeddings/query_metadata.json
[OK] Native per-query results: /content/drive/MyDrive/maxmin_research/results/mldr003_native_test_per_query.json
[OK] TEST queries: /content/mldr003/test.jsonl.gz
[OK] TEST qrels: /content/mldr003/qrels-test.tsv

Chunk embedding shape: (238301, 1024)
Query embedding shape: (400, 1024)
Chunk corpus type: list
Chunk corpus entries: 238301

STATUS: INPUTS VERIFIED


In [ ]:
import json
from pathlib import Path

CORPUS_PATH = (
    Path("/content/drive/MyDrive/maxmin_research")
    / "frozen/mldr003_maxmin_chunk_corpus.json"
)

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    chunk_corpus = json.load(f)

print("Number of chunks:", len(chunk_corpus))
print("First record type:", type(chunk_corpus[0]).__name__)
print("\nFirst chunk record:")
print(json.dumps(chunk_corpus[0], ensure_ascii=False, indent=2)[:2500])

if isinstance(chunk_corpus[0], dict):
    print("\nRecord keys:", list(chunk_corpus[0].keys()))

Number of chunks: 238301
First record type: dict

First chunk record:
{
  "doc_id": "doc-hi-428",
  "chunk_idx": 0,
  "sentence_indices": [
    0,
    1
  ],
  "text": "उज्जैन [[चित्र:Mahakal temple.jpg|250px|right|thumb|उज्जैन का प्रसिद्ध महाकालेश्वर मंदिर]] उज्जैन (उज्जयिनी) भारत के मध्य प्रदेश राज्य का एक प्रमुख शहर है जो क्षिप्रा नदी या शिप्रा नदी के किनारे पर बसा है। यह एक अत्यन्त प्राचीन शहर है।",
  "token_count": 80
}

Record keys: ['doc_id', 'chunk_idx', 'sentence_indices', 'text', 'token_count']


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
FROZEN = RESEARCH / "frozen"
MLDR_DIR = Path("/content/mldr003")
RESULTS = RESEARCH / "results"

# Load saved inputs
chunk_embeddings = np.load(
    CACHE / "maxmin_chunk_embeddings.npy",
    mmap_mode="r"
)
query_embeddings = np.load(CACHE / "query_embeddings.npy")

with open(CACHE / "query_metadata.json", "r", encoding="utf-8") as f:
    query_metadata = json.load(f)

with open(FROZEN / "mldr003_maxmin_chunk_corpus.json", "r", encoding="utf-8") as f:
    chunk_corpus = json.load(f)

# TEST queries and relevance labels
test_queries = {}
with gzip.open(MLDR_DIR / "test.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        test_queries[str(row["query_id"])] = row["query"]

qrels = {}
with open(MLDR_DIR / "qrels-test.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4 and int(parts[3]) > 0:
            qrels.setdefault(str(parts[0]), set()).add(parts[2])

query_row = {
    str(item["query_id"]): i
    for i, item in enumerate(query_metadata)
}

test_ids = sorted(test_queries)

assert len(test_ids) == 200
assert set(test_queries) == set(qrels)
assert all(qid in query_row for qid in test_ids)
assert chunk_embeddings.shape == (len(chunk_corpus), 1024)

# Prepare chunk parent IDs
chunk_parent_ids = np.array(
    [item["doc_id"] for item in chunk_corpus],
    dtype=object
)

# Build a list of unique parent IDs and chunk membership.
unique_parent_ids = list(dict.fromkeys(chunk_parent_ids.tolist()))
parent_to_idx = {doc_id: i for i, doc_id in enumerate(unique_parent_ids)}
chunk_parent_idx = np.array(
    [parent_to_idx[doc_id] for doc_id in chunk_parent_ids],
    dtype=np.int32
)

# Exact chunk scoring in query batches; max-score collapse to parent docs.
per_query = []

print("Running Max-Min retrieval for 200 TEST queries...")

chunk_matrix = np.asarray(chunk_embeddings, dtype=np.float32)

for count, qid in enumerate(test_ids, start=1):
    qi = query_row[qid]
    qvec = query_embeddings[qi].astype(np.float32)

    chunk_scores = chunk_matrix @ qvec

    # Parent score = maximum score of any chunk belonging to that parent.
    parent_scores = np.full(len(unique_parent_ids), -np.inf, dtype=np.float32)
    np.maximum.at(parent_scores, chunk_parent_idx, chunk_scores)

    top_parent_idx = np.argsort(-parent_scores)[:10]
    ranked_ids = [unique_parent_ids[i] for i in top_parent_idx]

    relevant = qrels[qid]
    hits = np.array([1 if doc_id in relevant else 0 for doc_id in ranked_ids])

    discounts = 1.0 / np.log2(np.arange(2, 12))
    ndcg = float(np.sum(hits * discounts))  # One relevant doc per query
    hit_positions = np.flatnonzero(hits)
    mrr = 1.0 / (int(hit_positions[0]) + 1) if len(hit_positions) else 0.0

    relevant_rank = next(
        (i + 1 for i, doc_id in enumerate(ranked_ids) if doc_id in relevant),
        None
    )

    per_query.append({
        "query_id": qid,
        "ndcg@10": ndcg,
        "mrr@10": mrr,
        "recall@1": float(hits[0]),
        "recall@10": float(np.any(hits)),
        "relevant_rank": relevant_rank,
        "top10_doc_ids": ranked_ids
    })

    if count % 25 == 0 or count == len(test_ids):
        print(f"Completed: {count}/200")

metrics = {
    "n_queries": len(per_query),
    "ndcg@10": float(np.mean([r["ndcg@10"] for r in per_query])),
    "mrr@10": float(np.mean([r["mrr@10"] for r in per_query])),
    "recall@1": float(np.mean([r["recall@1"] for r in per_query])),
    "recall@10": float(np.mean([r["recall@10"] for r in per_query]))
}

out_path = RESULTS / "mldr003_maxmin_test_per_query_recheck.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {"metrics": metrics, "per_query": per_query},
        f, ensure_ascii=False, indent=2
    )

print("\n" + "=" * 55)
print("REPRODUCED MAX-MIN TEST RESULTS")
print("=" * 55)

for metric, value in metrics.items():
    print(f"{metric}: {value:.6f}" if isinstance(value, float)
          else f"{metric}: {value}")

print("\nSaved:", out_path)
print("STATUS: COMPLETE")

Running Max-Min retrieval for 200 TEST queries...
Completed: 25/200
Completed: 50/200
Completed: 75/200
Completed: 100/200
Completed: 125/200
Completed: 150/200
Completed: 175/200
Completed: 200/200

REPRODUCED MAX-MIN TEST RESULTS
n_queries: 200
ndcg@10: 0.567445
mrr@10: 0.536520
recall@1: 0.475000
recall@10: 0.665000

Saved: /content/drive/MyDrive/maxmin_research/results/mldr003_maxmin_test_per_query_recheck.json
STATUS: COMPLETE


In [ ]:
import json
import numpy as np
from pathlib import Path

RESULTS = Path("/content/drive/MyDrive/maxmin_research/results")

original_path = RESULTS / "mldr003_maxmin_test_per_query.json"
recheck_path = RESULTS / "mldr003_maxmin_test_per_query_recheck.json"

print("Original file exists:", original_path.exists())
print("Recheck file exists:", recheck_path.exists())

if original_path.exists():
    with open(original_path, "r", encoding="utf-8") as f:
        original_data = json.load(f)

    with open(recheck_path, "r", encoding="utf-8") as f:
        recheck_data = json.load(f)

    # Support either a wrapped {"per_query": [...]} format or a plain list.
    original_rows = (
        original_data["per_query"]
        if isinstance(original_data, dict) and "per_query" in original_data
        else original_data
    )
    recheck_rows = (
        recheck_data["per_query"]
        if isinstance(recheck_data, dict) and "per_query" in recheck_data
        else recheck_data
    )

    original = {str(r["query_id"]): r for r in original_rows}
    recheck = {str(r["query_id"]): r for r in recheck_rows}

    print("\nOriginal query count:", len(original))
    print("Recheck query count:", len(recheck))
    print("Same query IDs:", set(original) == set(recheck))

    if set(original) == set(recheck):
        changed = []

        for qid in sorted(original):
            a, b = original[qid], recheck[qid]

            if (
                a.get("relevant_rank") != b.get("relevant_rank")
                or a.get("top10_doc_ids") != b.get("top10_doc_ids")
            ):
                changed.append({
                    "query_id": qid,
                    "original_rank": a.get("relevant_rank"),
                    "recheck_rank": b.get("relevant_rank"),
                    "original_top10": a.get("top10_doc_ids"),
                    "recheck_top10": b.get("top10_doc_ids"),
                })

        print("Queries with different rankings:", len(changed))

        for row in changed[:10]:
            print("\n", json.dumps(row, ensure_ascii=False, indent=2))
    else:
        print("Cannot compare: query-ID sets differ.")
else:
    print(
        "\nThe original per-query file is missing. "
        "We can still validate the recheck against the original aggregate "
        "metrics, but cannot identify query-level differences."
    )

Original file exists: False
Recheck file exists: True

The original per-query file is missing. We can still validate the recheck against the original aggregate metrics, but cannot identify query-level differences.


In [ ]:
import json
import numpy as np
from pathlib import Path

RESULTS = Path("/content/drive/MyDrive/maxmin_research/results")

with open(RESULTS / "mldr003_native_test_per_query.json", encoding="utf-8") as f:
    native_data = json.load(f)

with open(RESULTS / "mldr003_maxmin_test_per_query_recheck.json", encoding="utf-8") as f:
    maxmin_data = json.load(f)

native_rows = (
    native_data["per_query"]
    if isinstance(native_data, dict) and "per_query" in native_data
    else native_data
)
maxmin_rows = (
    maxmin_data["per_query"]
    if isinstance(maxmin_data, dict) and "per_query" in maxmin_data
    else maxmin_data
)

native = {str(r["query_id"]): r for r in native_rows}
maxmin = {str(r["query_id"]): r for r in maxmin_rows}

assert len(native) == 200, f"Expected 200 Native queries, got {len(native)}"
assert len(maxmin) == 200, f"Expected 200 Max-Min queries, got {len(maxmin)}"
assert set(native) == set(maxmin), "Query IDs do not match!"

metrics = ["ndcg@10", "mrr@10", "recall@1", "recall@10"]

print("Matched queries:", len(native))
print("\nMetric comparison")
print("-" * 66)
print(f"{'Metric':<15}{'Native':>12}{'Max-Min':>12}{'Delta':>12}")
print("-" * 66)

for metric in metrics:
    n = np.array([native[q][metric] for q in sorted(native)], dtype=float)
    m = np.array([maxmin[q][metric] for q in sorted(maxmin)], dtype=float)

    print(f"{metric:<15}{n.mean():>12.6f}{m.mean():>12.6f}{(m-n).mean():>12.6f}")

print("\nSTATUS: PAIRED QUERY SET VERIFIED")

Matched queries: 200

Metric comparison
------------------------------------------------------------------
Metric               Native     Max-Min       Delta
------------------------------------------------------------------
ndcg@10            0.409589    0.567445    0.157857
mrr@10             0.372262    0.536520    0.164258
recall@1           0.310000    0.475000    0.165000
recall@10          0.530000    0.665000    0.135000

STATUS: PAIRED QUERY SET VERIFIED


In [ ]:
import json
import numpy as np
from pathlib import Path

RESULTS = Path("/content/drive/MyDrive/maxmin_research/results")

with open(RESULTS / "mldr003_native_test_per_query.json", encoding="utf-8") as f:
    native_data = json.load(f)

with open(RESULTS / "mldr003_maxmin_test_per_query_recheck.json", encoding="utf-8") as f:
    maxmin_data = json.load(f)

native_rows = native_data["per_query"] if isinstance(native_data, dict) else native_data
maxmin_rows = maxmin_data["per_query"] if isinstance(maxmin_data, dict) else maxmin_data

native = {str(r["query_id"]): r for r in native_rows}
maxmin = {str(r["query_id"]): r for r in maxmin_rows}

assert len(native) == len(maxmin) == 200
assert set(native) == set(maxmin)

query_ids = sorted(native)
metrics = ["ndcg@10", "mrr@10", "recall@1", "recall@10"]

rng = np.random.default_rng(20261011)
n_queries = len(query_ids)
n_boot = 10_000

print("Paired bootstrap: Max-Min minus Native")
print(f"Queries: {n_queries} | Resamples: {n_boot} | Seed: 20261011")
print()

for metric in metrics:
    native_values = np.array(
        [native[q][metric] for q in query_ids], dtype=float
    )
    maxmin_values = np.array(
        [maxmin[q][metric] for q in query_ids], dtype=float
    )

    paired_delta = maxmin_values - native_values
    observed = float(paired_delta.mean())

    # Resample query indices; both methods remain paired within each query.
    indices = rng.integers(0, n_queries, size=(n_boot, n_queries))
    bootstrap_deltas = paired_delta[indices].mean(axis=1)

    ci_low, ci_high = np.percentile(bootstrap_deltas, [2.5, 97.5])

    print(f"{metric}")
    print(f"  Native mean:  {native_values.mean():.6f}")
    print(f"  Max-Min mean: {maxmin_values.mean():.6f}")
    print(f"  Delta:        {observed:+.6f}")
    print(f"  95% CI:       [{ci_low:+.6f}, {ci_high:+.6f}]")
    print(f"  CI excludes zero: {ci_low > 0 or ci_high < 0}")
    print()

print("STATUS: BOOTSTRAP COMPLETE")

Paired bootstrap: Max-Min minus Native
Queries: 200 | Resamples: 10000 | Seed: 20261011

ndcg@10
  Native mean:  0.409589
  Max-Min mean: 0.567445
  Delta:        +0.157857
  95% CI:       [+0.098099, +0.218374]
  CI excludes zero: True

mrr@10
  Native mean:  0.372262
  Max-Min mean: 0.536520
  Delta:        +0.164258
  95% CI:       [+0.100856, +0.228736]
  CI excludes zero: True

recall@1
  Native mean:  0.310000
  Max-Min mean: 0.475000
  Delta:        +0.165000
  95% CI:       [+0.095000, +0.240000]
  CI excludes zero: True

recall@10
  Native mean:  0.530000
  Max-Min mean: 0.665000
  Delta:        +0.135000
  95% CI:       [+0.065000, +0.205000]
  CI excludes zero: True

STATUS: BOOTSTRAP COMPLETE


In [ ]:
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
RESULTS = RESEARCH / "results"

NATIVE_PATH = RESULTS / "mldr003_native_test_per_query.json"
MAXMIN_PATH = RESULTS / "mldr003_maxmin_test_per_query_recheck.json"
ORIGINAL_MAXMIN_PATH = RESULTS / "mldr003_maxmin_test_per_query.json"

def load_rows(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "per_query" in data:
        rows = data["per_query"]
    elif isinstance(data, list):
        rows = data
    else:
        raise ValueError(f"Unexpected JSON structure: {path}")

    return data, rows

def audit_result_file(path, label):
    data, rows = load_rows(path)
    qids = [str(r["query_id"]) for r in rows]

    assert len(rows) == 200, f"{label}: expected 200 rows, got {len(rows)}"
    assert len(set(qids)) == 200, f"{label}: duplicate query IDs"

    print(f"\n{label}")
    print("-" * 45)
    print("Rows:", len(rows))
    print("Unique query IDs:", len(set(qids)))

    for metric in ["ndcg@10", "mrr@10", "recall@1", "recall@10"]:
        values = np.array([r[metric] for r in rows], dtype=float)
        print(f"{metric}: {values.mean():.9f}")

    return data, rows, set(qids)

assert NATIVE_PATH.exists(), f"Missing: {NATIVE_PATH}"
assert MAXMIN_PATH.exists(), f"Missing: {MAXMIN_PATH}"

native_data, native_rows, native_qids = audit_result_file(
    NATIVE_PATH, "NATIVE"
)
maxmin_data, maxmin_rows, maxmin_qids = audit_result_file(
    MAXMIN_PATH, "MAX-MIN RECHECK"
)

print("\nCROSS-METHOD CHECKS")
print("-" * 45)
print("Same 200 query IDs:", native_qids == maxmin_qids)

assert native_qids == maxmin_qids, "Native and Max-Min query sets differ"

for label, rows in [("Native", native_rows), ("Max-Min", maxmin_rows)]:
    for r in rows:
        ranked = r.get("top10_doc_ids", [])
        assert len(ranked) <= 10, f"{label}: ranking longer than 10"
        assert len(ranked) == len(set(ranked)), (
            f"{label}: duplicate parent document in ranking for "
            f"query {r['query_id']}"
        )

print("No duplicate parent documents in stored top-10 rankings.")

print("\nORIGINAL MAX-MIN FILE")
print("-" * 45)
print("Original per-query file exists:", ORIGINAL_MAXMIN_PATH.exists())

if isinstance(maxmin_data, dict) and "metrics" in maxmin_data:
    print("Stored aggregate matches recomputed metrics:")
    for metric in ["ndcg@10", "mrr@10", "recall@1", "recall@10"]:
        recomputed = float(np.mean([r[metric] for r in maxmin_rows]))
        stored = float(maxmin_data["metrics"][metric])
        print(
            f"{metric}: stored={stored:.9f}, "
            f"recomputed={recomputed:.9f}, "
            f"match={np.isclose(stored, recomputed, atol=1e-10)}"
        )

print("\nSTATUS: AUDIT COMPLETE")


NATIVE
---------------------------------------------
Rows: 200
Unique query IDs: 200
ndcg@10: 0.409588642
mrr@10: 0.372261905
recall@1: 0.310000000
recall@10: 0.530000000

MAX-MIN RECHECK
---------------------------------------------
Rows: 200
Unique query IDs: 200
ndcg@10: 0.567445387
mrr@10: 0.536519841
recall@1: 0.475000000
recall@10: 0.665000000

CROSS-METHOD CHECKS
---------------------------------------------
Same 200 query IDs: True
No duplicate parent documents in stored top-10 rankings.

ORIGINAL MAX-MIN FILE
---------------------------------------------
Original per-query file exists: False
Stored aggregate matches recomputed metrics:
ndcg@10: stored=0.567445387, recomputed=0.567445387, match=True
mrr@10: stored=0.536519841, recomputed=0.536519841, match=True
recall@1: stored=0.475000000, recomputed=0.475000000, match=True
recall@10: stored=0.665000000, recomputed=0.665000000, match=True

STATUS: AUDIT COMPLETE


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
RESULTS = RESEARCH / "results"
MLDR_DIR = Path("/content/mldr003")

# Load rankings
with open(RESULTS / "mldr003_native_test_per_query.json", encoding="utf-8") as f:
    native_data = json.load(f)

with open(RESULTS / "mldr003_maxmin_test_per_query_recheck.json", encoding="utf-8") as f:
    maxmin_data = json.load(f)

native_rows = native_data["per_query"] if isinstance(native_data, dict) else native_data
maxmin_rows = maxmin_data["per_query"] if isinstance(maxmin_data, dict) else maxmin_data

# Independently reload ground-truth relevance labels
qrels = {}
with open(MLDR_DIR / "qrels-test.tsv", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4 and int(parts[3]) > 0:
            qrels.setdefault(str(parts[0]), set()).add(parts[2])

assert len(qrels) == 200
assert all(len(rels) == 1 for rels in qrels.values())

def recompute_metrics(rows, method):
    results = []

    for row in rows:
        qid = str(row["query_id"])
        assert qid in qrels, f"Missing qrels for {qid}"

        ranked = row["top10_doc_ids"]
        relevant = qrels[qid]

        hits = np.array(
            [1.0 if doc_id in relevant else 0.0 for doc_id in ranked]
        )

        # One relevant document per query, binary relevance.
        discounts = 1.0 / np.log2(np.arange(2, len(hits) + 2))
        ndcg = float(np.sum(hits * discounts))

        hit_positions = np.flatnonzero(hits)
        mrr = (
            1.0 / (int(hit_positions[0]) + 1)
            if len(hit_positions) else 0.0
        )

        relevant_rank = (
            int(hit_positions[0]) + 1 if len(hit_positions) else None
        )

        results.append({
            "query_id": qid,
            "ndcg@10": ndcg,
            "mrr@10": mrr,
            "recall@1": float(len(hits) > 0 and hits[0] == 1.0),
            "recall@10": float(len(hit_positions) > 0),
            "relevant_rank": relevant_rank,
        })

    metrics = {
        metric: float(np.mean([r[metric] for r in results]))
        for metric in ["ndcg@10", "mrr@10", "recall@1", "recall@10"]
    }

    print(f"\n{method} — recomputed directly from rankings and qrels")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.9f}")

    return results, metrics

native_verified, native_metrics = recompute_metrics(native_rows, "NATIVE")
maxmin_verified, maxmin_metrics = recompute_metrics(maxmin_rows, "MAX-MIN")

print("\nCOMPARISON: MAX-MIN MINUS NATIVE")
for metric in native_metrics:
    delta = maxmin_metrics[metric] - native_metrics[metric]
    print(f"{metric}: {delta:+.9f}")

# Verify each independently recomputed metric against its saved value.
for rows, verified, label in [
    (native_rows, native_verified, "Native"),
    (maxmin_rows, maxmin_verified, "Max-Min"),
]:
    saved = {str(r["query_id"]): r for r in rows}
    for result in verified:
        original = saved[result["query_id"]]
        for metric in ["ndcg@10", "mrr@10", "recall@1", "recall@10"]:
            assert np.isclose(
                result[metric], original[metric], atol=1e-10
            ), (
                f"{label} mismatch for {result['query_id']} "
                f"{metric}: stored={original[metric]}, "
                f"recomputed={result[metric]}"
            )

print("\nSTATUS: RANKING METRICS VERIFIED AGAINST QRELS")


NATIVE — recomputed directly from rankings and qrels
ndcg@10: 0.409588642
mrr@10: 0.372261905
recall@1: 0.310000000
recall@10: 0.530000000

MAX-MIN — recomputed directly from rankings and qrels
ndcg@10: 0.567445387
mrr@10: 0.536519841
recall@1: 0.475000000
recall@10: 0.665000000

COMPARISON: MAX-MIN MINUS NATIVE
ndcg@10: +0.157856745
mrr@10: +0.164257937
recall@1: +0.165000000
recall@10: +0.135000000

STATUS: RANKING METRICS VERIFIED AGAINST QRELS


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
RESULTS = RESEARCH / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

canonical = {
    "experiment": "003",
    "dataset": "MLDR Hindi",
    "split": "TEST",
    "n_queries": 200,
    "embedding_model": "BAAI/bge-m3",
    "retrieval_protocol": {
        "native": "One embedding per parent document; inputs truncated to the model's 8192-token limit.",
        "maxmin": "One embedding per Max-Min chunk; parent score is the maximum similarity among its chunks.",
        "query_pool": "Same 200 TEST queries for both methods.",
        "relevance": "Original MLDR Hindi TEST qrels; one relevant document per query."
    },
    "verified_metrics": {
        "native": {
            "ndcg@10": 0.409588642,
            "mrr@10": 0.372261905,
            "recall@1": 0.310000000,
            "recall@10": 0.530000000
        },
        "maxmin": {
            "ndcg@10": 0.567445387,
            "mrr@10": 0.536519841,
            "recall@1": 0.475000000,
            "recall@10": 0.665000000
        },
        "delta_maxmin_minus_native": {
            "ndcg@10": 0.157856745,
            "mrr@10": 0.164257937,
            "recall@1": 0.165000000,
            "recall@10": 0.135000000
        }
    },
    "paired_query_bootstrap": {
        "resamples": 10000,
        "seed": 20261011,
        "confidence_level": 0.95,
        "interval_type": "Percentile bootstrap",
        "delta_maxmin_minus_native": {
            "ndcg@10": [0.098099, 0.218374],
            "mrr@10": [0.100856, 0.228736],
            "recall@1": [0.095000, 0.240000],
            "recall@10": [0.065000, 0.205000]
        },
        "note": "Intervals copied from the completed bootstrap output; rounded to six decimals."
    },
    "verification": {
        "same_200_query_ids": True,
        "unique_query_ids_per_method": True,
        "duplicate_parent_ids_in_top10": False,
        "metrics_recomputed_from_rankings_and_qrels": True,
        "recomputed_metrics_match_saved_per_query_metrics": True
    },
    "reproducibility_note": {
        "original_maxmin_aggregate": {
            "ndcg@10": 0.569291,
            "mrr@10": 0.539020,
            "recall@1": 0.480000,
            "recall@10": 0.665000
        },
        "canonical_maxmin_aggregate": {
            "ndcg@10": 0.567445387,
            "mrr@10": 0.536519841,
            "recall@1": 0.475000,
            "recall@10": 0.665000
        },
        "explanation": "The original per-query result file is missing. The earlier aggregate discrepancy could not be traced to individual queries and remains unresolved. Canonical metrics were independently recomputed from the available saved rankings and qrels."
    },
    "canonical_source_files": {
        "native_per_query": str(RESULTS / "mldr003_native_test_per_query.json"),
        "maxmin_per_query": str(RESULTS / "mldr003_maxmin_test_per_query_recheck.json")
    },
    "frozen_at_utc": datetime.now(timezone.utc).isoformat()
}

output_path = RESULTS / "mldr003_canonical_verified_results.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(canonical, f, ensure_ascii=False, indent=2)

# Read it back to confirm the record is valid JSON.
with open(output_path, "r", encoding="utf-8") as f:
    check = json.load(f)

assert check["verification"]["metrics_recomputed_from_rankings_and_qrels"]
assert check["n_queries"] == 200

print("CANONICAL RESULTS FROZEN")
print("File:", output_path)
print("Native per-query source:", canonical["canonical_source_files"]["native_per_query"])
print("Max-Min per-query source:", canonical["canonical_source_files"]["maxmin_per_query"])
print("Verification: PASS")
print("Original aggregate discrepancy: DOCUMENTED, UNRESOLVED")

CANONICAL RESULTS FROZEN
File: /content/drive/MyDrive/maxmin_research/results/mldr003_canonical_verified_results.json
Native per-query source: /content/drive/MyDrive/maxmin_research/results/mldr003_native_test_per_query.json
Max-Min per-query source: /content/drive/MyDrive/maxmin_research/results/mldr003_maxmin_test_per_query_recheck.json
Verification: PASS
Original aggregate discrepancy: DOCUMENTED, UNRESOLVED


In [ ]:
import gzip
import json
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
RESULTS = RESEARCH / "results"
MLDR_DIR = Path("/content/mldr003")

# 1. Load corpus using its actual schema: docid, text
corpus = {}

with gzip.open(MLDR_DIR / "corpus.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        corpus[str(row["docid"])] = row["text"]

print("Corpus documents:", len(corpus))
assert len(corpus) == 3806

# 2. Load TEST qrels
qrels = {}

with open(MLDR_DIR / "qrels-test.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4 and int(parts[3]) > 0:
            qrels.setdefault(str(parts[0]), set()).add(str(parts[2]))

assert len(qrels) == 200
assert all(len(docs) == 1 for docs in qrels.values())

# 3. Load saved retrieval results
with open(RESULTS / "mldr003_native_test_per_query.json", encoding="utf-8") as f:
    native_data = json.load(f)

with open(RESULTS / "mldr003_maxmin_test_per_query_recheck.json", encoding="utf-8") as f:
    maxmin_data = json.load(f)

native_rows = native_data["per_query"]
maxmin_rows = maxmin_data["per_query"]

native = {str(r["query_id"]): r for r in native_rows}
maxmin = {str(r["query_id"]): r for r in maxmin_rows}

assert len(native) == len(maxmin) == 200
assert set(native) == set(maxmin) == set(qrels)

# 4. Load BGE-M3 tokenizer
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

# 5. Calculate relevant-document length and per-query retrieval difference
records = []

for qid in sorted(qrels):
    relevant_doc_id = next(iter(qrels[qid]))
    relevant_text = corpus[relevant_doc_id]

    doc_tokens = len(
        tokenizer(
            relevant_text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    n_score = float(native[qid]["ndcg@10"])
    m_score = float(maxmin[qid]["ndcg@10"])

    records.append({
        "query_id": qid,
        "relevant_doc_id": relevant_doc_id,
        "relevant_doc_tokens": doc_tokens,
        "native_ndcg10": n_score,
        "maxmin_ndcg10": m_score,
        "delta_ndcg10": m_score - n_score,
        "native_relevant_rank": native[qid].get("relevant_rank"),
        "maxmin_relevant_rank": maxmin[qid].get("relevant_rank"),
        "relevant_doc_exceeds_8192": doc_tokens > 8192
    })

length_df = pd.DataFrame(records)

print("\nLENGTH ANALYSIS PREPARED")
print("Queries:", len(length_df))
print("Unique relevant documents:", length_df["relevant_doc_id"].nunique())
print("Missing lengths:", int(length_df["relevant_doc_tokens"].isna().sum()))

print("\nRelevant-document token length distribution:")
print(
    length_df["relevant_doc_tokens"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .round(1)
    .to_string()
)

print("\nRelevant documents exceeding 8192 tokens:")
print(
    length_df["relevant_doc_exceeds_8192"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nMean paired nDCG@10 difference (Max-Min - Native):")
print(f"{length_df['delta_ndcg10'].mean():+.6f}")

print("\nSTATUS: LENGTH DATA PREPARED")

Corpus documents: 3806

LENGTH ANALYSIS PREPARED
Queries: 200
Unique relevant documents: 200
Missing lengths: 0

Relevant-document token length distribution:
count     200.0
mean     6303.1
std       868.4
min      5023.0
25%      5514.0
50%      6279.0
75%      6980.5
90%      7593.9
95%      7782.2
max      7967.0

Relevant documents exceeding 8192 tokens:
relevant_doc_exceeds_8192
False    200

Mean paired nDCG@10 difference (Max-Min - Native):
+0.157857

STATUS: LENGTH DATA PREPARED


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Use the length_df produced by Cell 1.
df = length_df.copy()

# Split into four approximately equal-sized groups by relevant-document length.
df["length_quartile"] = pd.qcut(
    df["relevant_doc_tokens"],
    q=4,
    labels=["Q1 (shortest)", "Q2", "Q3", "Q4 (longest)"],
    duplicates="drop"
)

print("PERFORMANCE BY RELEVANT-DOCUMENT LENGTH")
print("=" * 88)

summary = (
    df.groupby("length_quartile", observed=True)
      .agg(
          queries=("query_id", "count"),
          min_tokens=("relevant_doc_tokens", "min"),
          median_tokens=("relevant_doc_tokens", "median"),
          max_tokens=("relevant_doc_tokens", "max"),
          native_ndcg10=("native_ndcg10", "mean"),
          maxmin_ndcg10=("maxmin_ndcg10", "mean"),
          mean_delta=("delta_ndcg10", "mean"),
      )
      .reset_index()
)

# Recall@10 reconstructed from the saved relevant-document ranks.
df["native_recall10"] = df["native_relevant_rank"].notna().astype(float)
df["maxmin_recall10"] = df["maxmin_relevant_rank"].notna().astype(float)

recall_summary = (
    df.groupby("length_quartile", observed=True)
      .agg(
          native_recall10=("native_recall10", "mean"),
          maxmin_recall10=("maxmin_recall10", "mean")
      )
      .reset_index()
)

summary = summary.merge(recall_summary, on="length_quartile")

print(summary.to_string(
    index=False,
    formatters={
        "min_tokens": "{:.0f}".format,
        "median_tokens": "{:.0f}".format,
        "max_tokens": "{:.0f}".format,
        "native_ndcg10": "{:.4f}".format,
        "maxmin_ndcg10": "{:.4f}".format,
        "mean_delta": "{:+.4f}".format,
        "native_recall10": "{:.3f}".format,
        "maxmin_recall10": "{:.3f}".format,
    }
))

# Association between relevant-document length and paired improvement.
rho, p_value = spearmanr(
    df["relevant_doc_tokens"],
    df["delta_ndcg10"]
)

print("\nLENGTH–IMPROVEMENT ASSOCIATION")
print("-" * 45)
print(f"Spearman rho: {rho:+.4f}")
print(f"Two-sided p-value: {p_value:.6f}")
print(f"Queries: {len(df)}")

print("\nInterpretation guide:")
print("- Positive rho: longer relevant documents tend to show larger Max-Min gains.")
print("- Near-zero rho: no clear monotonic relationship with length.")
print("- Negative rho: gains tend to be smaller for longer relevant documents.")

print("\nSTATUS: LENGTH-STRATIFIED ANALYSIS COMPLETE")

PERFORMANCE BY RELEVANT-DOCUMENT LENGTH
length_quartile  queries min_tokens median_tokens max_tokens native_ndcg10 maxmin_ndcg10 mean_delta native_recall10 maxmin_recall10
  Q1 (shortest)       50       5023          5252       5511        0.2878        0.5111    +0.2233           0.400           0.640
             Q2       50       5515          5877       6273        0.3159        0.4986    +0.1828           0.460           0.560
             Q3       50       6285          6579       6974        0.4714        0.6737    +0.2024           0.640           0.760
   Q4 (longest)       50       7000          7472       7967        0.5634        0.5863    +0.0230           0.620           0.700

LENGTH–IMPROVEMENT ASSOCIATION
---------------------------------------------
Spearman rho: -0.1493
Two-sided p-value: 0.034823
Queries: 200

Interpretation guide:
- Positive rho: longer relevant documents tend to show larger Max-Min gains.
- Near-zero rho: no clear monotonic relationship with lengt

In [ ]:
import gzip
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

MLDR_DIR = Path("/content/mldr003")
RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
RESULTS = RESEARCH / "results"

# Load corpus using its verified schema: docid, text
corpus = {}
with gzip.open(MLDR_DIR / "corpus.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        corpus[str(row["docid"])] = row["text"]

# Find the DEV query and qrels files
dev_query_path = MLDR_DIR / "dev.jsonl.gz"
dev_qrels_path = MLDR_DIR / "qrels-dev.tsv"

print("DEV query file exists:", dev_query_path.exists())
print("DEV qrels file exists:", dev_qrels_path.exists())

if not dev_query_path.exists() or not dev_qrels_path.exists():
    print("\nDEV files are missing from the current runtime.")
    print("We need to download them before continuing.")
else:
    # Load DEV qrels
    dev_qrels = {}
    with open(dev_qrels_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 4 and int(parts[3]) > 0:
                dev_qrels.setdefault(str(parts[0]), set()).add(str(parts[2]))

    # Load DEV queries
    dev_queries = {}
    with gzip.open(dev_query_path, "rt", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            dev_queries[str(row["query_id"])] = row["query"]

    assert set(dev_queries) == set(dev_qrels), (
        "DEV query IDs and qrels IDs do not match."
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

    # Measure relevant-document lengths
    rows = []
    missing_docs = []

    for qid in sorted(dev_qrels):
        for doc_id in sorted(dev_qrels[qid]):
            if doc_id not in corpus:
                missing_docs.append(doc_id)
                continue

            n_tokens = len(
                tokenizer(
                    corpus[doc_id],
                    add_special_tokens=True,
                    truncation=False
                )["input_ids"]
            )

            rows.append({
                "query_id": qid,
                "doc_id": doc_id,
                "tokens": n_tokens,
                "exceeds_8192": n_tokens > 8192
            })

    dev_length_df = pd.DataFrame(rows)

    print("\nDEV DATASET AUDIT")
    print("-" * 45)
    print("Corpus documents:", len(corpus))
    print("DEV queries:", len(dev_queries))
    print("DEV qrels query IDs:", len(dev_qrels))
    print("Positive query-document pairs:", sum(map(len, dev_qrels.values())))
    print("Relevant documents measured:", len(dev_length_df))
    print("Missing relevant documents:", len(missing_docs))
    print("Unique relevant documents:", dev_length_df["doc_id"].nunique())

    print("\nRelevant-document length distribution:")
    print(
        dev_length_df["tokens"]
        .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
        .round(1)
        .to_string()
    )

    print("\nRelevant documents over the 8192-token limit:")
    print(dev_length_df["exceeds_8192"].value_counts().to_string())

    print("\nSAVED ARTIFACTS")
    print("-" * 45)
    for path in [
        CACHE / "query_embeddings.npy",
        CACHE / "native_embeddings.npy",
        CACHE / "maxmin_chunk_embeddings.npy",
        RESEARCH / "frozen" / "mldr003_maxmin_chunk_corpus.json",
    ]:
        print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

    print("\nSTATUS: DEV AUDIT COMPLETE")

DEV query file exists: False
DEV qrels file exists: False

DEV files are missing from the current runtime.
We need to download them before continuing.


In [ ]:
import requests
from pathlib import Path

MLDR_DIR = Path("/content/mldr003")
MLDR_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "dev.jsonl.gz": (
        "https://huggingface.co/datasets/Shitao/MLDR/resolve/main/"
        "mldr-v1.0-hi/dev.jsonl.gz?download=true"
    ),
    "qrels-dev.tsv": (
        "https://huggingface.co/datasets/Shitao/MLDR/resolve/main/"
        "qrels/qrels.mldr-v1.0-hi-dev.tsv?download=true"
    ),
}

for filename, url in files.items():
    destination = MLDR_DIR / filename

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    destination.write_bytes(response.content)
    print(f"Downloaded {filename}: {destination.stat().st_size:,} bytes")

# Basic integrity checks
import gzip
import json

with gzip.open(MLDR_DIR / "dev.jsonl.gz", "rt", encoding="utf-8") as f:
    dev_query_count = sum(1 for _ in f)

with open(MLDR_DIR / "qrels-dev.tsv", "r", encoding="utf-8") as f:
    qrel_lines = [line for line in f if line.strip()]

print("\nDEV queries:", dev_query_count)
print("DEV qrels rows:", len(qrel_lines))

assert dev_query_count == 200, f"Expected 200 DEV queries, got {dev_query_count}"
assert len(qrel_lines) > 0, "DEV qrels file is empty"

print("STATUS: DEV FILES DOWNLOADED AND CHECKED")

Downloaded dev.jsonl.gz: 10,159,598 bytes
Downloaded qrels-dev.tsv: 5,161 bytes

DEV queries: 200
DEV qrels rows: 200
STATUS: DEV FILES DOWNLOADED AND CHECKED


In [ ]:
import gzip
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

MLDR_DIR = Path("/content/mldr003")

# Load corpus
corpus = {}
with gzip.open(MLDR_DIR / "corpus.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        corpus[str(row["docid"])] = row["text"]

# Load DEV queries
dev_queries = {}
with gzip.open(MLDR_DIR / "dev.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        dev_queries[str(row["query_id"])] = row["query"]

# Load DEV qrels
dev_qrels = {}
with open(MLDR_DIR / "qrels-dev.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4 and int(parts[3]) > 0:
            dev_qrels.setdefault(str(parts[0]), set()).add(str(parts[2]))

assert set(dev_queries) == set(dev_qrels)
assert len(dev_queries) == 200

# Tokenize relevant documents
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
rows = []

for qid in sorted(dev_qrels):
    for doc_id in sorted(dev_qrels[qid]):
        if doc_id not in corpus:
            raise KeyError(f"Relevant document missing from corpus: {doc_id}")

        token_count = len(
            tokenizer(
                corpus[doc_id],
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "tokens": token_count,
            "exceeds_8192": token_count > 8192
        })

dev_length_df = pd.DataFrame(rows)

print("DEV relevant-document audit")
print("=" * 50)
print("Queries:", len(dev_queries))
print("Positive query-document pairs:", len(dev_length_df))
print("Unique relevant documents:", dev_length_df["doc_id"].nunique())

print("\nToken-length distribution:")
print(
    dev_length_df["tokens"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .round(1)
    .to_string()
)

print("\nRelevant-document counts by 8192-token threshold:")
print(
    dev_length_df["exceeds_8192"]
    .value_counts()
    .rename(index={False: "At most 8192 tokens", True: "Over 8192 tokens"})
    .to_string()
)

print("\nLongest relevant documents:")
print(
    dev_length_df.nlargest(10, "tokens")[
        ["query_id", "doc_id", "tokens", "exceeds_8192"]
    ].to_string(index=False)
)

print("\nSTATUS: DEV LENGTH AUDIT COMPLETE")

DEV relevant-document audit
Queries: 200
Positive query-document pairs: 200
Unique relevant documents: 200

Token-length distribution:
count     200.0
mean     6287.6
std       868.7
min      5002.0
25%      5483.8
50%      6235.0
75%      7048.2
90%      7533.4
95%      7700.7
max      7988.0

Relevant-document counts by 8192-token threshold:
exceeds_8192
At most 8192 tokens    200

Longest relevant documents:
 query_id      doc_id  tokens  exceeds_8192
 q-hi-477 doc-hi-3361    7988         False
 q-hi-129  doc-hi-975    7900         False
 q-hi-796 doc-hi-5098    7861         False
q-hi-1023 doc-hi-6189    7838         False
 q-hi-741 doc-hi-4823    7831         False
 q-hi-607 doc-hi-4110    7819         False
 q-hi-896 doc-hi-2901    7817         False
 q-hi-656 doc-hi-3768    7790         False
 q-hi-994 doc-hi-6072    7761         False
 q-hi-150 doc-hi-1148    7734         False

STATUS: DEV LENGTH AUDIT COMPLETE


In [ ]:
from pathlib import Path
import numpy as np
import json

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
FROZEN = RESEARCH / "frozen"

paths = {
    "Query embeddings": CACHE / "query_embeddings.npy",
    "Native document embeddings": CACHE / "native_embeddings.npy",
    "Max-Min chunk embeddings": CACHE / "maxmin_chunk_embeddings.npy",
    "Query metadata": CACHE / "query_metadata.json",
    "Max-Min chunk corpus": FROZEN / "mldr003_maxmin_chunk_corpus.json",
}

print("SAVED ARTIFACT AUDIT")
print("=" * 60)

for label, path in paths.items():
    print(f"\n{label}: {'EXISTS' if path.exists() else 'MISSING'}")
    if path.exists():
        print(f"  Size: {path.stat().st_size / (1024**2):.2f} MiB")

for label, path in [
    ("Query embeddings", paths["Query embeddings"]),
    ("Native document embeddings", paths["Native document embeddings"]),
    ("Max-Min chunk embeddings", paths["Max-Min chunk embeddings"]),
]:
    if path.exists():
        arr = np.load(path, mmap_mode="r")
        print(f"\n{label} shape: {arr.shape}")

if paths["Query metadata"].exists():
    with open(paths["Query metadata"], encoding="utf-8") as f:
        metadata = json.load(f)

    from collections import Counter
    print("\nQuery metadata split counts:")
    print(Counter(item["split"] for item in metadata))

print("\nSTATUS: ARTIFACT AUDIT COMPLETE")

SAVED ARTIFACT AUDIT

Query embeddings: EXISTS
  Size: 1.56 MiB

Native document embeddings: EXISTS
  Size: 14.87 MiB

Max-Min chunk embeddings: EXISTS
  Size: 930.86 MiB

Query metadata: EXISTS
  Size: 0.11 MiB

Max-Min chunk corpus: EXISTS
  Size: 189.86 MiB

Query embeddings shape: (400, 1024)

Native document embeddings shape: (3806, 1024)

Max-Min chunk embeddings shape: (238301, 1024)

Query metadata split counts:
Counter({'dev': 200, 'test': 200})

STATUS: ARTIFACT AUDIT COMPLETE


In [ ]:
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"

with open(CACHE / "query_metadata.json", encoding="utf-8") as f:
    query_metadata = json.load(f)

query_embeddings = np.load(CACHE / "query_embeddings.npy", mmap_mode="r")
native_embeddings = np.load(CACHE / "native_embeddings.npy", mmap_mode="r")
maxmin_embeddings = np.load(CACHE / "maxmin_chunk_embeddings.npy", mmap_mode="r")

# Explicit mapping: never assume DEV/TEST occupy particular row ranges.
query_row = {
    str(item["query_id"]): i
    for i, item in enumerate(query_metadata)
}

dev_ids = {
    str(item["query_id"])
    for item in query_metadata
    if item["split"].lower() == "dev"
}
test_ids = {
    str(item["query_id"])
    for item in query_metadata
    if item["split"].lower() == "test"
}

assert len(query_metadata) == 400
assert len(query_row) == 400
assert len(dev_ids) == 200
assert len(test_ids) == 200
assert dev_ids.isdisjoint(test_ids)

assert query_embeddings.shape == (400, 1024)
assert native_embeddings.shape == (3806, 1024)
assert maxmin_embeddings.shape == (238301, 1024)

# Check that embeddings are finite and approximately unit-normalized.
sample_indices = np.linspace(
    0, len(query_embeddings) - 1, 40, dtype=int
)
sample_norms = np.linalg.norm(
    np.asarray(query_embeddings[sample_indices], dtype=np.float32),
    axis=1
)

assert np.isfinite(sample_norms).all()
assert np.all(np.abs(sample_norms - 1.0) < 0.002)

print("DEV query IDs:", len(dev_ids))
print("TEST query IDs:", len(test_ids))
print("Split overlap:", len(dev_ids & test_ids))
print("Query mapping complete:", all(qid in query_row for qid in dev_ids))
print("Embedding shapes verified.")
print(f"Sampled query norm range: {sample_norms.min():.6f}–{sample_norms.max():.6f}")
print("STATUS: READY FOR DEV RETRIEVAL")

DEV query IDs: 200
TEST query IDs: 200
Split overlap: 0
Query mapping complete: True
Embedding shapes verified.
Sampled query norm range: 0.999712–1.000261
STATUS: READY FOR DEV RETRIEVAL


In [ ]:
import gzip
import json
import numpy as np
from pathlib import Path

RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
CACHE = RESEARCH / "cache" / "mldr003_embeddings"
RESULTS = RESEARCH / "results"
FROZEN = RESEARCH / "frozen"
MLDR_DIR = Path("/content/mldr003")

# Load saved embeddings and metadata
query_embeddings = np.load(CACHE / "query_embeddings.npy")
native_embeddings = np.load(CACHE / "native_embeddings.npy", mmap_mode="r")
maxmin_embeddings = np.load(CACHE / "maxmin_chunk_embeddings.npy", mmap_mode="r")

with open(CACHE / "query_metadata.json", encoding="utf-8") as f:
    query_metadata = json.load(f)

with open(FROZEN / "mldr003_maxmin_chunk_corpus.json", encoding="utf-8") as f:
    chunk_corpus = json.load(f)

query_row = {
    str(item["query_id"]): i
    for i, item in enumerate(query_metadata)
}

# Load DEV queries and qrels
dev_queries = {}

with gzip.open(MLDR_DIR / "dev.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        dev_queries[str(row["query_id"])] = row["query"]

dev_qrels = {}

with open(MLDR_DIR / "qrels-dev.tsv", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 4 and int(parts[3]) > 0:
            dev_qrels.setdefault(str(parts[0]), set()).add(parts[2])

dev_ids = sorted(dev_queries)

assert len(dev_ids) == 200
assert set(dev_ids) == set(dev_qrels)
assert all(qid in query_row for qid in dev_ids)
assert native_embeddings.shape == (3806, 1024)
assert maxmin_embeddings.shape == (len(chunk_corpus), 1024)

# Parent-document mapping for Max-Min chunks
chunk_parent_ids = [str(item["doc_id"]) for item in chunk_corpus]
unique_parent_ids = list(dict.fromkeys(chunk_parent_ids))
parent_to_idx = {doc_id: i for i, doc_id in enumerate(unique_parent_ids)}
chunk_parent_idx = np.array(
    [parent_to_idx[doc_id] for doc_id in chunk_parent_ids],
    dtype=np.int32
)

# Native corpus document ordering must match the embedding construction.
# Recover it from the original corpus, whose ordering was used to build embeddings.
corpus = {}
with gzip.open(MLDR_DIR / "corpus.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        corpus[str(row["docid"])] = row["text"]

native_doc_ids = list(corpus.keys())
assert len(native_doc_ids) == native_embeddings.shape[0]

# Load query embedding vectors; use existing model embeddings.
native_results = []
maxmin_results = []

# Use block-wise chunk scoring to avoid a large per-query temporary matrix.
CHUNK_BLOCK_SIZE = 20000
maxmin_matrix = np.asarray(maxmin_embeddings, dtype=np.float32)

for count, qid in enumerate(dev_ids, start=1):
    qvec = np.asarray(query_embeddings[query_row[qid]], dtype=np.float32)

    # Native: parent-document scores
    native_scores = np.asarray(native_embeddings, dtype=np.float32) @ qvec
    native_top_idx = np.argsort(-native_scores)[:10]
    native_top_ids = [native_doc_ids[i] for i in native_top_idx]

    # Max-Min: score chunks in blocks, then aggregate max score by parent.
    parent_scores = np.full(len(unique_parent_ids), -np.inf, dtype=np.float32)

    for start in range(0, len(chunk_corpus), CHUNK_BLOCK_SIZE):
        end = min(start + CHUNK_BLOCK_SIZE, len(chunk_corpus))
        block_scores = maxmin_matrix[start:end] @ qvec
        np.maximum.at(
            parent_scores,
            chunk_parent_idx[start:end],
            block_scores
        )

    maxmin_top_idx = np.argsort(-parent_scores)[:10]
    maxmin_top_ids = [unique_parent_ids[i] for i in maxmin_top_idx]

    for method, ranked_ids, output in [
        ("native", native_top_ids, native_results),
        ("maxmin", maxmin_top_ids, maxmin_results)
    ]:
        relevant = dev_qrels[qid]
        hits = np.array([1.0 if doc_id in relevant else 0.0 for doc_id in ranked_ids])
        discounts = 1.0 / np.log2(np.arange(2, 12))

        positions = np.flatnonzero(hits)
        rank = int(positions[0]) + 1 if len(positions) else None

        output.append({
            "query_id": qid,
            "ndcg@10": float(np.sum(hits * discounts)),
            "mrr@10": float(1.0 / rank) if rank is not None else 0.0,
            "recall@1": float(hits[0]),
            "recall@10": float(len(positions) > 0),
            "relevant_rank": rank,
            "top10_doc_ids": ranked_ids
        })

    if count % 25 == 0:
        print(f"Completed DEV queries: {count}/200")

def aggregate(rows):
    return {
        "n_queries": len(rows),
        **{
            metric: float(np.mean([r[metric] for r in rows]))
            for metric in ["ndcg@10", "mrr@10", "recall@1", "recall@10"]
        }
    }

native_metrics = aggregate(native_results)
maxmin_metrics = aggregate(maxmin_results)

for method, metrics, rows in [
    ("native", native_metrics, native_results),
    ("maxmin", maxmin_metrics, maxmin_results)
]:
    out_path = RESULTS / f"mldr003_{method}_dev_per_query.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(
            {"metrics": metrics, "per_query": rows},
            f, ensure_ascii=False, indent=2
        )

print("\nDEV RESULTS")
print("=" * 65)
print(f"{'Metric':<15}{'Native':>12}{'Max-Min':>12}{'Delta':>12}")

for metric in ["ndcg@10", "mrr@10", "recall@1", "recall@10"]:
    delta = maxmin_metrics[metric] - native_metrics[metric]
    print(
        f"{metric:<15}{native_metrics[metric]:>12.6f}"
        f"{maxmin_metrics[metric]:>12.6f}{delta:>+12.6f}"
    )

print("\nSaved DEV per-query results.")
print("STATUS: DEV RETRIEVAL COMPLETE")

Completed DEV queries: 25/200
Completed DEV queries: 50/200
Completed DEV queries: 75/200
Completed DEV queries: 100/200
Completed DEV queries: 125/200
Completed DEV queries: 150/200
Completed DEV queries: 175/200
Completed DEV queries: 200/200

DEV RESULTS
Metric               Native     Max-Min       Delta
ndcg@10            0.395547    0.526520   +0.130974
mrr@10             0.361649    0.496651   +0.135002
recall@1           0.300000    0.435000   +0.135000
recall@10          0.505000    0.620000   +0.115000

Saved DEV per-query results.
STATUS: DEV RETRIEVAL COMPLETE


In [ ]:
import gzip
import json
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

MLDR_DIR = Path("/content/mldr003")
RESEARCH = Path("/content/drive/MyDrive/maxmin_research")
RESULTS = RESEARCH / "results"

# Load the corpus
corpus = {}
with gzip.open(MLDR_DIR / "corpus.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        corpus[str(row["docid"])] = row["text"]

# Load tokenizer and measure full document lengths
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

doc_lengths = {}
for doc_id, text in corpus.items():
    doc_lengths[doc_id] = len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

long_doc_ids = {
    doc_id for doc_id, length in doc_lengths.items()
    if length > 8192
}

print("CORPUS LENGTH AUDIT")
print("=" * 60)
print("Corpus documents:", len(corpus))
print("Documents >8192 tokens:", len(long_doc_ids))
print(f"Share of corpus: {len(long_doc_ids) / len(corpus):.2%}")

# Load relevance judgments
def load_qrels(path):
    qrels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 4 and int(parts[3]) > 0:
                qrels.setdefault(str(parts[0]), set()).add(str(parts[2]))
    return qrels

dev_qrels = load_qrels(MLDR_DIR / "qrels-dev.tsv")
test_qrels = load_qrels(MLDR_DIR / "qrels-test.tsv")

def summarize_qrels(qrels, label):
    all_relevant = set().union(*qrels.values())
    long_relevant = all_relevant & long_doc_ids
    queries_with_long_relevant = {
        qid for qid, docs in qrels.items()
        if docs & long_doc_ids
    }

    print(f"\n{label} relevance coverage")
    print("-" * 45)
    print("Queries:", len(qrels))
    print("Unique relevant documents:", len(all_relevant))
    print("Unique relevant documents >8192 tokens:", len(long_relevant))
    print("Queries with at least one long relevant document:",
          len(queries_with_long_relevant))

    return long_relevant, queries_with_long_relevant

dev_long_relevant, dev_long_queries = summarize_qrels(dev_qrels, "DEV")
test_long_relevant, test_long_queries = summarize_qrels(test_qrels, "TEST")

# Inspect existing top-10 rankings
def load_rows(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["per_query"] if isinstance(data, dict) else data

ranking_files = {
    "DEV Native": RESULTS / "mldr003_native_dev_per_query.json",
    "DEV Max-Min": RESULTS / "mldr003_maxmin_dev_per_query.json",
    "TEST Native": RESULTS / "mldr003_native_test_per_query.json",
    "TEST Max-Min": RESULTS / "mldr003_maxmin_test_per_query_recheck.json",
}

print("\nLONG DOCUMENTS IN SAVED TOP-10 RANKINGS")
print("=" * 60)

for label, path in ranking_files.items():
    if not path.exists():
        print(f"{label}: result file missing")
        continue

    rows = load_rows(path)
    retrieved_long_ids = {
        str(doc_id)
        for row in rows
        for doc_id in row.get("top10_doc_ids", [])
        if str(doc_id) in long_doc_ids
    }

    query_hits = sum(
        any(str(doc_id) in long_doc_ids for doc_id in row.get("top10_doc_ids", []))
        for row in rows
    )

    print(f"\n{label}")
    print("Queries retrieving >=1 long document in top 10:", query_hits)
    print("Unique long documents retrieved:", len(retrieved_long_ids))

print("\nSTATUS: LONG-DOCUMENT COVERAGE AUDIT COMPLETE")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8943 > 8192). Running this sequence through the model will result in indexing errors


CORPUS LENGTH AUDIT
Corpus documents: 3806
Documents >8192 tokens: 955
Share of corpus: 25.09%

DEV relevance coverage
---------------------------------------------
Queries: 200
Unique relevant documents: 200
Unique relevant documents >8192 tokens: 0
Queries with at least one long relevant document: 0

TEST relevance coverage
---------------------------------------------
Queries: 200
Unique relevant documents: 200
Unique relevant documents >8192 tokens: 0
Queries with at least one long relevant document: 0

LONG DOCUMENTS IN SAVED TOP-10 RANKINGS

DEV Native
Queries retrieving >=1 long document in top 10: 199
Unique long documents retrieved: 571

DEV Max-Min
Queries retrieving >=1 long document in top 10: 200
Unique long documents retrieved: 515

TEST Native
Queries retrieving >=1 long document in top 10: 199
Unique long documents retrieved: 569

TEST Max-Min
Queries retrieving >=1 long document in top 10: 197
Unique long documents retrieved: 531

STATUS: LONG-DOCUMENT COVERAGE AUDIT C

In [ ]:
import gzip
import json
from pathlib import Path

corpus_path = Path("/content/mldr003/corpus.jsonl.gz")

with gzip.open(corpus_path, "rt", encoding="utf-8") as f:
    first_record = json.loads(next(f))

print("Available keys:", list(first_record.keys()))
print("\nFirst record:")
print(json.dumps(first_record, ensure_ascii=False, indent=2)[:2000])

Available keys: ['docid', 'text']

First record:
{
  "docid": "doc-hi-428",
  "text": "उज्जैन\n[[चित्र:Mahakal temple.jpg|250px|right|thumb|उज्जैन का प्रसिद्ध महाकालेश्वर मंदिर]]\nउज्जैन (उज्जयिनी) भारत के मध्य प्रदेश राज्य का एक प्रमुख शहर है जो क्षिप्रा नदी या शिप्रा नदी के किनारे पर बसा है। यह एक अत्यन्त प्राचीन शहर है। यह महान सम्राट विक्रमादित्य के राज्य की राजधानी थी ।  उज्जैन को कालिदास की नगरी के नाम से भी जाना जाता है। यहाँ हर 12 वर्ष पर सिंहस्थ महाकुंभ मेला लगता है। भगवान शिव के 12 ज्योतिर्लिंगों में एक महाकाल इस नगरी में स्थित है। उज्जैन मध्य प्रदेश के सबसे बड़े शहर इन्दौर से 45 कि॰मी॰ पर है। उज्जैन के प्राचीन नाम अवन्तिका, उज्जयनी, कनकश्रन्गा आदि है। उज्जैन मंदिरों की नगरी है। यहाँ कई तीर्थ स्थल है। इसकी जनसंख्या 515215  लाख सन ,2011 की जनगणना के हिसाब से  है। यह मध्य प्रदेश का पाँचवा सबसे बड़ा शहर है। नगर निगम सीमा का क्षेत्रफल 152 वर्ग किलोमीटर है।\n\nदर्शनीय स्थल\nचिन्तामन\tगणेश\nस्थिरमन\tगणेश\nगढ़कालिका\tमाताजी\nहरसिद्धि\tमाताजी\nकालभैरव\tदेव\nमहाकालेश्वर\tमहादेव\nसिद्ध